# Notebook 4 — Revenue Leakage Risk

## Objective

This notebook develops a forward-looking model of customer revenue leakage.

Earlier exploratory analysis showed that identified-customer revenue leakage was driven primarily by:

- customers becoming inactive;
- customers remaining active but spending materially less;
- and only partially offsetting growth from expanding, reactivated and newly acquired customers.

The objective of this notebook is therefore to identify customers who are currently active but are at elevated risk of generating substantially less revenue during a future observation window.

This is not a permanent-churn model.

A customer may leak revenue because they:

- stop purchasing temporarily;
- reduce order frequency;
- reduce basket value;
- reduce purchase volume;
- or shift into a lower-value purchasing pattern.

The modelling task is designed around future customer revenue deterioration rather than permanent customer loss.

## Revenue Leakage Target

For each eligible customer prediction point:

- **Baseline window:** prior 90 days
- **Prediction date:** end of the baseline window
- **Outcome window:** following 90 days
- **Baseline revenue:** positive merchandise revenue generated during the prior 90 days
- **Future revenue:** positive merchandise revenue generated during the following 90 days

A customer is classified as a revenue-leakage event when:

> **future 90-day revenue ≤ 50% of prior 90-day revenue**

This includes customers who become fully inactive and customers whose spending contracts materially.

The target is designed to measure economically meaningful deterioration rather than permanent churn.

### Why Use a 50% Revenue-Decline Threshold?

A leakage model requires a materiality threshold.

A very small decline would classify normal purchasing volatility as leakage.

A very extreme threshold would identify only customers who effectively disappear.

A 50% decline provides a practical middle ground that captures substantial deterioration while still including customers who remain active at much lower value.

The threshold will later be sensitivity-tested against alternative definitions such as:

- 25% revenue decline;
- 40% revenue decline;
- 60% revenue decline;
- and complete inactivity.

The primary model will use the 50% decline definition unless diagnostics indicate that the target is excessively rare or unstable.

In [ ]:
# ================================================================
# 39.4 — IMPORTS
# ================================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ================================================================
# 39.5 — LOAD CANONICAL CLEANED DATA
# ================================================================

df = pd.read_parquet(
    "cleaned_online_retail.parquet"
)

print(
    f"Rows loaded: {len(df):,}"
)

print(
    f"Columns loaded: {df.shape[1]:,}"
)

In [ ]:
# ================================================================
# 39.6 — REQUIRED COLUMN VALIDATION
# ================================================================

required_columns = [
    "date",
    "invoice",
    "customer",
    "product",
    "region",
    "quantity",
    "unit_price",
    "customer_is_identified",
    "is_merchandise",
    "is_cancellation",
    "net_merchandise_revenue",
]


missing_required_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


assert not missing_required_columns, (
    f"Missing columns: {missing_required_columns}"
)


print(
    "✓ Required columns confirmed."
)

In [ ]:
# ================================================================
# 39.7 — MODELLING END DATE
# ================================================================

MODELLING_END_DATE = pd.Timestamp(
    "2011-11-30 23:59:59"
)


print(
    f"Modelling observation boundary: "
    f"{MODELLING_END_DATE}"
)

In [ ]:
# ================================================================
# 39.8 — QUALIFYING CUSTOMER PURCHASES
# ================================================================

positive_customer_purchases = (
    df.loc[
        df[
            "customer_is_identified"
        ]
        &
        df[
            "is_merchandise"
        ]
        &
        (
            df[
                "quantity"
            ]
            > 0
        )
        &
        (
            df[
                "unit_price"
            ]
            > 0
        )
        &
        (
            ~df[
                "is_cancellation"
            ]
        )
    ]
    .copy()
)


positive_customer_purchases = (
    positive_customer_purchases.loc[
        positive_customer_purchases[
            "date"
        ]
        <=
        MODELLING_END_DATE
    ]
    .copy()
)


print(
    f"Qualifying rows: "
    f"{len(positive_customer_purchases):,}"
)

print(
    f"Identified customers: "
    f"{positive_customer_purchases['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 39.9 — CUSTOMER-DAY AGGREGATION
# ================================================================

customer_daily = (
    positive_customer_purchases
    .groupby(
        [
            "customer",
            pd.Grouper(
                key="date",
                freq="D"
            )
        ],
        as_index=False
    )
    .agg(
        daily_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        daily_units=(
            "quantity",
            "sum"
        ),

        daily_orders=(
            "invoice",
            "nunique"
        ),

        daily_products=(
            "product",
            "nunique"
        ),

        region=(
            "region",
            "first"
        ),
    )
)


print(
    f"Customer-day rows: "
    f"{len(customer_daily):,}"
)

### 39.10 — Create candidate prediction dates
To keep the model operationally realistic and manageable, use monthly prediction points.

I will work with month-end prediction dates.

Example:
- predict at 31 Mar
- prior baseline = Jan–Mar
- future outcome = Apr–Jun

This avoids creating one prediction every day for every customer and makes temporal validation cleaner.

In [ ]:
# ================================================================
# 39.10 — MONTH-END PREDICTION DATES
# ================================================================

PREDICTION_START_DATE = pd.Timestamp(
    "2010-04-30"
)


PREDICTION_END_DATE = pd.Timestamp(
    "2011-08-31"
)


prediction_dates = pd.date_range(
    start=PREDICTION_START_DATE,
    end=PREDICTION_END_DATE,
    freq="ME"
)


print(
    f"Prediction dates: "
    f"{len(prediction_dates)}"
)

print(
    prediction_dates
)

Why end in Aug 2011?

Because each prediction needs a full 90-day future outcome window before the end-November modelling boundary.

In [ ]:
# ================================================================
# 39.11 — WINDOW DEFINITIONS
# ================================================================

BASELINE_WINDOW_DAYS = 90
OUTCOME_WINDOW_DAYS = 90
LEAKAGE_THRESHOLD = 0.50


print(
    f"Baseline window: "
    f"{BASELINE_WINDOW_DAYS} days"
)

print(
    f"Outcome window: "
    f"{OUTCOME_WINDOW_DAYS} days"
)

print(
    f"Leakage threshold: "
    f"{LEAKAGE_THRESHOLD:.0%} of baseline revenue"
)

### 39.12 — Build customer prediction episodes
We will iterate across monthly prediction dates and create one row per eligible active customer.

Eligibility:
- customer has positive revenue during previous 90 days.

In [ ]:
# ================================================================
# 39.12 — BUILD CUSTOMER REVENUE LEAKAGE EPISODES
# ================================================================

leakage_episode_records = []


for prediction_date in prediction_dates:

    baseline_start = (
        prediction_date
        -
        pd.Timedelta(
            days=BASELINE_WINDOW_DAYS
        )
    )


    outcome_end = (
        prediction_date
        +
        pd.Timedelta(
            days=OUTCOME_WINDOW_DAYS
        )
    )


    # ------------------------------------------------------------
    # Baseline window
    # ------------------------------------------------------------

    baseline_window = (
        customer_daily.loc[
            (
                customer_daily[
                    "date"
                ]
                >
                baseline_start
            )
            &
            (
                customer_daily[
                    "date"
                ]
                <=
                prediction_date
            )
        ]
    )


    baseline_summary = (
        baseline_window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            baseline_revenue=(
                "daily_revenue",
                "sum"
            ),

            baseline_orders=(
                "daily_orders",
                "sum"
            ),

            baseline_active_days=(
                "date",
                "nunique"
            ),

            baseline_units=(
                "daily_units",
                "sum"
            ),

            baseline_product_days=(
                "daily_products",
                "sum"
            ),

            last_region=(
                "region",
                "last"
            ),
        )
    )


    # ------------------------------------------------------------
    # Keep customers active in baseline
    # ------------------------------------------------------------

    baseline_summary = (
        baseline_summary.loc[
            baseline_summary[
                "baseline_revenue"
            ]
            > 0
        ]
        .copy()
    )


    # ------------------------------------------------------------
    # Future outcome window
    # ------------------------------------------------------------

    outcome_window = (
        customer_daily.loc[
            (
                customer_daily[
                    "date"
                ]
                >
                prediction_date
            )
            &
            (
                customer_daily[
                    "date"
                ]
                <=
                outcome_end
            )
        ]
    )


    outcome_summary = (
        outcome_window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            future_revenue=(
                "daily_revenue",
                "sum"
            ),

            future_orders=(
                "daily_orders",
                "sum"
            ),

            future_active_days=(
                "date",
                "nunique"
            ),
        )
    )


    # ------------------------------------------------------------
    # Merge
    # ------------------------------------------------------------

    prediction_population = (
        baseline_summary
        .merge(
            outcome_summary,
            on="customer",
            how="left",
            validate="one_to_one"
        )
    )


    prediction_population[
        [
            "future_revenue",
            "future_orders",
            "future_active_days",
        ]
    ] = (
        prediction_population[
            [
                "future_revenue",
                "future_orders",
                "future_active_days",
            ]
        ]
        .fillna(0)
    )


    prediction_population[
        "prediction_date"
    ] = (
        prediction_date
    )


    prediction_population[
        "baseline_start"
    ] = (
        baseline_start
    )


    prediction_population[
        "outcome_end"
    ] = (
        outcome_end
    )


    leakage_episode_records.append(
        prediction_population
    )

In [ ]:
revenue_leakage_episodes = pd.concat(
    leakage_episode_records,
    ignore_index=True
)


print(
    f"Candidate prediction episodes: "
    f"{len(revenue_leakage_episodes):,}"
)

print(
    f"Unique customers: "
    f"{revenue_leakage_episodes['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 39.13 — FUTURE REVENUE RETENTION RATIO
# ================================================================

revenue_leakage_episodes[
    "future_to_baseline_revenue_ratio"
] = (
    revenue_leakage_episodes[
        "future_revenue"
    ]
    /
    revenue_leakage_episodes[
        "baseline_revenue"
    ]
)

In [ ]:
# ================================================================
# 39.14 — PRIMARY REVENUE LEAKAGE TARGET
# ================================================================

revenue_leakage_episodes[
    "revenue_leakage_event"
] = (
    revenue_leakage_episodes[
        "future_to_baseline_revenue_ratio"
    ]
    <=
    LEAKAGE_THRESHOLD
).astype(int)

In [ ]:
# ================================================================
# 39.15 — REVENUE LEAKAGE AMOUNT
# ================================================================

revenue_leakage_episodes[
    "revenue_leakage_amount"
] = (
    revenue_leakage_episodes[
        "baseline_revenue"
    ]
    -
    revenue_leakage_episodes[
        "future_revenue"
    ]
)


revenue_leakage_episodes[
    "revenue_leakage_amount"
] = (
    revenue_leakage_episodes[
        "revenue_leakage_amount"
    ]
    .clip(
        lower=0
    )
)

In [ ]:
# ================================================================
# 39.16 — LEAKAGE TYPE
# ================================================================

revenue_leakage_episodes[
    "future_inactive"
] = (
    revenue_leakage_episodes[
        "future_revenue"
    ]
    ==
    0
).astype(int)


revenue_leakage_episodes[
    "material_contraction"
] = (
    (
        revenue_leakage_episodes[
            "future_revenue"
        ]
        > 0
    )
    &
    (
        revenue_leakage_episodes[
            "revenue_leakage_event"
        ]
        == 1
    )
).astype(int)

In [ ]:
# ================================================================
# 39.17 — PRIMARY TARGET SUMMARY
# ================================================================

leakage_target_summary = (
    revenue_leakage_episodes[
        "revenue_leakage_event"
    ]
    .value_counts(
        normalize=False
    )
    .rename_axis(
        "revenue_leakage_event"
    )
    .reset_index(
        name="episodes"
    )
)


leakage_target_summary[
    "share_pct"
] = (
    leakage_target_summary[
        "episodes"
    ]
    /
    leakage_target_summary[
        "episodes"
    ]
    .sum()
    *
    100
)


display(
    leakage_target_summary
)

In [ ]:
# ================================================================
# 39.18 — LEAKAGE MECHANISM SUMMARY
# ================================================================

leakage_mechanism_summary = pd.DataFrame(
    {
        "Mechanism": [
            "Future inactivity",
            "Material contraction while still active",
            "No material leakage",
        ],

        "Episodes": [
            revenue_leakage_episodes[
                "future_inactive"
            ].sum(),

            revenue_leakage_episodes[
                "material_contraction"
            ].sum(),

            (
                revenue_leakage_episodes[
                    "revenue_leakage_event"
                ]
                == 0
            ).sum(),
        ],
    }
)


leakage_mechanism_summary[
    "Share (%)"
] = (
    leakage_mechanism_summary[
        "Episodes"
    ]
    /
    len(
        revenue_leakage_episodes
    )
    *
    100
)


display(
    leakage_mechanism_summary
)

In [ ]:
# ================================================================
# 39.19 — LEAKAGE ECONOMICS
# ================================================================

revenue_leakage_economics = pd.DataFrame(
    {
        "Metric": [
            "Total baseline revenue",
            "Total future revenue",
            "Gross positive leakage amount",
            "Average leakage per leakage episode",
            "Median leakage per leakage episode",
        ],

        "Value": [
            revenue_leakage_episodes[
                "baseline_revenue"
            ].sum(),

            revenue_leakage_episodes[
                "future_revenue"
            ].sum(),

            revenue_leakage_episodes[
                "revenue_leakage_amount"
            ].sum(),

            revenue_leakage_episodes.loc[
                revenue_leakage_episodes[
                    "revenue_leakage_event"
                ]
                == 1,
                "revenue_leakage_amount"
            ].mean(),

            revenue_leakage_episodes.loc[
                revenue_leakage_episodes[
                    "revenue_leakage_event"
                ]
                == 1,
                "revenue_leakage_amount"
            ].median(),
        ],
    }
)


display(
    revenue_leakage_economics
)

In [ ]:
# ================================================================
# 39.20 — MONTHLY LEAKAGE RATE
# ================================================================

revenue_leakage_episodes[
    "prediction_period"
] = (
    revenue_leakage_episodes[
        "prediction_date"
    ]
    .dt.to_period("M")
)


monthly_leakage_summary = (
    revenue_leakage_episodes
    .groupby(
        "prediction_period",
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        unique_customers=(
            "customer",
            "nunique"
        ),

        leakage_rate=(
            "revenue_leakage_event",
            "mean"
        ),

        inactivity_rate=(
            "future_inactive",
            "mean"
        ),

        contraction_rate=(
            "material_contraction",
            "mean"
        ),

        average_baseline_revenue=(
            "baseline_revenue",
            "mean"
        ),

        average_future_revenue=(
            "future_revenue",
            "mean"
        ),
    )
)


monthly_leakage_summary[
    "leakage_rate_pct"
] = (
    monthly_leakage_summary[
        "leakage_rate"
    ]
    *
    100
)


display(
    monthly_leakage_summary
)

In [ ]:
plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=monthly_leakage_summary,
    x=monthly_leakage_summary[
        "prediction_period"
    ].astype(str),
    y="leakage_rate_pct",
    marker="o"
)


plt.title(
    "Revenue Leakage Rate by Prediction Month"
)

plt.xlabel(
    "Prediction Month"
)

plt.ylabel(
    "Revenue Leakage Rate (%)"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 39.21 — RETENTION-RATIO DISTRIBUTION
# ================================================================

revenue_retention_distribution = (
    revenue_leakage_episodes[
        "future_to_baseline_revenue_ratio"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)


display(
    revenue_retention_distribution
)

In [ ]:
plt.figure(
    figsize=(11, 6)
)


sns.histplot(
    revenue_leakage_episodes[
        "future_to_baseline_revenue_ratio"
    ]
    .clip(
        upper=3
    ),
    bins=50
)


plt.axvline(
    LEAKAGE_THRESHOLD,
    linestyle="--",
    label="Leakage threshold"
)


plt.title(
    "Future Revenue Relative to Prior 90-Day Revenue"
)

plt.xlabel(
    "Future Revenue / Baseline Revenue (visual capped at 3×)"
)

plt.ylabel(
    "Episodes"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 39.22 — TARGET DEFINITION SENSITIVITY
# ================================================================

leakage_thresholds = [
    0.25,
    0.40,
    0.50,
    0.60,
    0.75,
]


threshold_sensitivity_records = []


for threshold in leakage_thresholds:

    leakage_flag = (
        revenue_leakage_episodes[
            "future_to_baseline_revenue_ratio"
        ]
        <=
        threshold
    )


    threshold_sensitivity_records.append(
        {
            "Future Revenue Threshold":
                threshold,

            "Equivalent Revenue Decline":
                1
                -
                threshold,

            "Leakage Episodes":
                leakage_flag.sum(),

            "Leakage Rate":
                leakage_flag.mean(),

            "Leakage Rate (%)":
                leakage_flag.mean()
                *
                100,
        }
    )


leakage_threshold_sensitivity = pd.DataFrame(
    threshold_sensitivity_records
)


display(
    leakage_threshold_sensitivity
)

In [ ]:
# ================================================================
# 39.23 — REPEATED CUSTOMER EPISODES
# ================================================================

leakage_episodes_per_customer = (
    revenue_leakage_episodes
    .groupby(
        "customer"
    )
    .size()
)


leakage_episode_frequency_summary = (
    leakage_episodes_per_customer
    .describe()
)


display(
    leakage_episode_frequency_summary
)

In [ ]:
print(
    "Customers with multiple prediction episodes: "
    f"{(leakage_episodes_per_customer > 1).sum():,}"
)

print(
    "Maximum prediction episodes for one customer: "
    f"{leakage_episodes_per_customer.max():,}"
)

In [ ]:
# ================================================================
# 39.24 — LEAKAGE BY BASELINE REVENUE BAND
# ================================================================

revenue_leakage_episodes[
    "baseline_revenue_band"
] = pd.qcut(
    revenue_leakage_episodes[
        "baseline_revenue"
    ],
    q=5,
    labels=[
        "Q1 — Lowest",
        "Q2",
        "Q3",
        "Q4",
        "Q5 — Highest",
    ],
    duplicates="drop"
)


leakage_by_revenue_band = (
    revenue_leakage_episodes
    .groupby(
        "baseline_revenue_band",
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        median_baseline_revenue=(
            "baseline_revenue",
            "median"
        ),

        leakage_rate=(
            "revenue_leakage_event",
            "mean"
        ),

        future_inactivity_rate=(
            "future_inactive",
            "mean"
        ),

        average_leakage_amount=(
            "revenue_leakage_amount",
            "mean"
        ),
    )
)


leakage_by_revenue_band[
    "leakage_rate_pct"
] = (
    leakage_by_revenue_band[
        "leakage_rate"
    ]
    *
    100
)


display(
    leakage_by_revenue_band
)

## Initial Revenue Leakage Modelling Architecture

The first model will treat revenue leakage as a binary classification problem.

### Prediction Unit

One active customer at one monthly prediction date.

### Eligibility

The customer must have generated positive merchandise revenue during the previous 90 days.

### Prediction Point

Month-end.

### Feature Window

All information observable on or before the prediction date.

Initial features will include:

- baseline 90-day revenue;
- baseline order count;
- active purchase days;
- units purchased;
- product breadth;
- average order value;
- transaction frequency;
- recency within the baseline window;
- previous 90-day revenue;
- revenue trend;
- historical relationship depth;
- customer tenure;
- historical gap behaviour;
- prior leakage episodes;
- prior recoveries;
- region;
- and calendar features.

### Primary Target

Future 90-day revenue ≤ 50% of prior 90-day revenue.

### Secondary Outcomes

The notebook will separately monitor:

- complete future inactivity;
- material contraction among still-active customers;
- continuous revenue leakage amount;
- and future revenue retention ratio.

### Validation

All model validation will be chronological.

The final Test period will remain unopened until the model specification is frozen.


### Leakage-Control Rule

All future-window variables are target-construction fields only.

They must never be used as predictors.

This includes:

- future revenue;
- future orders;
- future active days;
- future-to-baseline revenue ratio;
- revenue leakage amount;
- future inactivity;
- material contraction;
- and any information observed after the prediction timestamp.

The model must only use information available when the prediction is made.

In [ ]:
# ================================================================
# 39.27 — STEP 39 COMPLETION CHECK
# ================================================================

assert (
    len(
        revenue_leakage_episodes
    )
    > 0
)


assert (
    revenue_leakage_episodes[
        "baseline_revenue"
    ]
    .gt(0)
    .all()
)


assert (
    revenue_leakage_episodes[
        "revenue_leakage_event"
    ]
    .isin(
        [0, 1]
    )
    .all()
)


assert (
    revenue_leakage_episodes[
        "outcome_end"
    ]
    .max()
    <=
    MODELLING_END_DATE
)


print(
    "✓ Step 39 complete."
)

print(
    "✓ Monthly active-customer prediction episodes created."
)

print(
    "✓ 90-day baseline and outcome windows created."
)

print(
    "✓ Primary 50% revenue-leakage target created."
)

print(
    "✓ Inactivity and contraction mechanisms separated."
)

print(
    "✓ Target sensitivity prepared."
)

### Step 39 Summary — Revenue Leakage Target Architecture

Step 39 converted customer revenue deterioration into a formal forward-looking prediction problem.

Each observation represents an identified customer who generated positive merchandise revenue during the preceding 90 days.

The primary target asks whether that customer's revenue during the following 90 days falls to **50% or less of the preceding 90-day baseline**.

### Primary Target Population

The initial dataset contains **35,321 active-customer prediction episodes**.

Approximately:

- **54.72%** are material revenue-leakage events;
- **45.28%** do not experience material leakage.

The target is therefore sufficiently balanced for ordinary binary classification without automatic resampling or class weighting.

### Leakage Is Primarily Driven by Future Inactivity

Among the 19,326 primary leakage events:

- approximately **79.5%** correspond to complete future inactivity;
- approximately **20.5%** correspond to material contraction among customers who remain active.

Revenue leakage is therefore strongly related to inactivity, but the target is broader than a churn-style classification because it also captures substantial deterioration among continuing customers.

### Revenue Risk and Revenue Exposure Are Different

Leakage probability declines as baseline customer value increases.

The lowest baseline-revenue quintile experienced approximately **65.6% leakage**, while the highest quintile experienced approximately **45.3% leakage**.

However, the financial amount exposed moves in the opposite direction.

Average episode-level leakage was approximately **84.9** in the lowest-value quintile compared with approximately **1,241.7** in the highest-value quintile.

A probability-only ranking would therefore not necessarily represent the economically optimal customer priority.

The final commercial architecture should eventually combine:

> **probability of leakage × financial exposure**

rather than relying on leakage probability alone.

### Strong Temporal and Seasonal Variation Exists

Monthly leakage rates vary materially.

Rates exceed 70% around November and December 2010 but decline toward approximately 40% in later summer periods.

This reflects meaningful calendar and customer-behaviour variation and may also include regression from unusually high seasonal baseline expenditure.

Random Train/Test splitting would therefore be inappropriate.

All modelling must preserve chronological order, and calendar features must be treated carefully.

### The 50% Decline Threshold Is Retained

Sensitivity analysis showed a gradual change in target prevalence across alternative leakage definitions.

The 50% future-revenue threshold produces a material but non-extreme event definition and retains a balanced modelling population.

The primary target will therefore remain:

> **future 90-day revenue ≤ 50% of prior 90-day revenue**

Alternative thresholds will remain available for later sensitivity analysis.

### Accounting Interpretation

Revenue leakage amounts in this notebook are episode-level analytical quantities.

Because the same customer can appear at multiple monthly prediction dates and the 90-day windows overlap, episode-level leakage amounts must not be summed and interpreted as unique annual company revenue loss.

The modelling objective is customer prioritisation, not financial-statement revenue reconciliation.


## Step 40 — Leakage-Safe Feature Engineering and Temporal Split

Step 40 converts the revenue-leakage episodes into a leakage-safe machine-learning dataset.

The feature architecture is designed around customer information genuinely observable at each month-end prediction point.

The principal predictors will describe:

- current 90-day revenue;
- previous 90-day revenue;
- recent 30-day behaviour;
- order frequency;
- customer activity;
- product breadth;
- transaction cadence;
- short-term revenue trend;
- prior leakage history;
- customer region;
- and calendar seasonality.

### Historical Observation Requirement

Several important predictors compare the current 90-day window with the immediately preceding 90-day window.

The primary modelling population therefore requires at least **180 days of source-data history before the prediction point**.

Prediction episodes without a complete 180-day historical window will be excluded from primary modelling.

### Temporal Validation Design

The leakage target itself requires 90 days of future observation.

To reduce overlap between model-development periods, deliberately unused temporal gaps will be placed between:

- Train and Validation;
- and Validation and Test.

This creates a more demanding approximation of genuine out-of-time generalisation.

The final Test target will not be inspected until the model specification has been frozen.

In [ ]:
# ================================================================
# 40.2 — NORMALISED PURCHASE DAY
# ================================================================

positive_customer_purchases[
    "purchase_day"
] = (
    positive_customer_purchases[
        "date"
    ]
    .dt.normalize()
)


customer_daily[
    "date"
] = (
    pd.to_datetime(
        customer_daily[
            "date"
        ]
    )
    .dt.normalize()
)

In [ ]:
# ================================================================
# 40.3 — COMPLETE-HISTORY REQUIREMENT
# ================================================================

SOURCE_START_DAY = (
    customer_daily[
        "date"
    ]
    .min()
    .normalize()
)


HISTORICAL_LOOKBACK_DAYS = 180


revenue_leakage_episodes[
    "has_complete_180d_history"
] = (
    (
        revenue_leakage_episodes[
            "prediction_date"
        ]
        -
        pd.Timedelta(
            days=HISTORICAL_LOOKBACK_DAYS
        )
    )
    >=
    SOURCE_START_DAY
)


history_coverage_summary = (
    revenue_leakage_episodes[
        "has_complete_180d_history"
    ]
    .value_counts()
    .rename_axis(
        "has_complete_180d_history"
    )
    .reset_index(
        name="episodes"
    )
)


display(
    history_coverage_summary
)

In [ ]:
# ================================================================
# 40.4 — PRIMARY MODELLING POPULATION
# ================================================================

leakage_model_episodes = (
    revenue_leakage_episodes.loc[
        revenue_leakage_episodes[
            "has_complete_180d_history"
        ]
    ]
    .copy()
)


print(
    f"Primary modelling episodes: "
    f"{len(leakage_model_episodes):,}"
)

print(
    f"Prediction period: "
    f"{leakage_model_episodes['prediction_period'].min()} "
    f"to "
    f"{leakage_model_episodes['prediction_period'].max()}"
)

In [ ]:
# ================================================================
# 40.5 — WINDOW SUMMARY HELPER
# ================================================================

def summarise_customer_window(
    daily_data,
    customers,
    start_date,
    end_date,
    prefix
):

    window = (
        daily_data.loc[
            daily_data[
                "customer"
            ].isin(
                customers
            )
            &
            (
                daily_data[
                    "date"
                ]
                >
                start_date
            )
            &
            (
                daily_data[
                    "date"
                ]
                <=
                end_date
            )
        ]
        .copy()
    )


    summary = (
        window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            revenue=(
                "daily_revenue",
                "sum"
            ),

            orders=(
                "daily_orders",
                "sum"
            ),

            active_days=(
                "date",
                "nunique"
            ),

            units=(
                "daily_units",
                "sum"
            ),

            first_purchase_day=(
                "date",
                "min"
            ),

            last_purchase_day=(
                "date",
                "max"
            ),
        )
    )


    summary = summary.rename(
        columns={
            column:
                f"{prefix}_{column}"
            for column in summary.columns
            if column != "customer"
        }
    )


    return summary

In [ ]:
# ================================================================
# 40.6 — PRODUCT-BREADTH HELPER
# ================================================================

def summarise_product_window(
    line_data,
    customers,
    start_date,
    end_date,
    prefix
):

    window = (
        line_data.loc[
            line_data[
                "customer"
            ].isin(
                customers
            )
            &
            (
                line_data[
                    "purchase_day"
                ]
                >
                start_date
            )
            &
            (
                line_data[
                    "purchase_day"
                ]
                <=
                end_date
            )
        ]
    )


    summary = (
        window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            unique_products=(
                "product",
                "nunique"
            ),

            average_unit_price=(
                "unit_price",
                "mean"
            ),

            maximum_unit_price=(
                "unit_price",
                "max"
            ),
        )
    )


    summary = summary.rename(
        columns={
            column:
                f"{prefix}_{column}"
            for column in summary.columns
            if column != "customer"
        }
    )


    return summary

In [ ]:
# ================================================================
# 40.7 — PURCHASE-GAP HELPER
# ================================================================

def summarise_purchase_gaps(
    daily_data,
    customers,
    start_date,
    end_date,
    prefix
):

    window = (
        daily_data.loc[
            daily_data[
                "customer"
            ].isin(
                customers
            )
            &
            (
                daily_data[
                    "date"
                ]
                >
                start_date
            )
            &
            (
                daily_data[
                    "date"
                ]
                <=
                end_date
            )
        ][
            [
                "customer",
                "date"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "customer",
                "date"
            ]
        )
        .copy()
    )


    window[
        "gap_days"
    ] = (
        window
        .groupby(
            "customer"
        )[
            "date"
        ]
        .diff()
        .dt.days
    )


    summary = (
        window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            average_gap_days=(
                "gap_days",
                "mean"
            ),

            maximum_gap_days=(
                "gap_days",
                "max"
            ),

            minimum_gap_days=(
                "gap_days",
                "min"
            ),
        )
    )


    summary = summary.rename(
        columns={
            column:
                f"{prefix}_{column}"
            for column in summary.columns
            if column != "customer"
        }
    )


    return summary

### 40.8 — Build historical features at each prediction date
We will use four windows:
- current 90 days: b90
- previous 90 days: p90
- most recent 30 days: r30
- full previous 180 days: h180

In [ ]:
# ================================================================
# 40.8 — BUILD LEAKAGE-SAFE HISTORICAL FEATURES
# ================================================================

feature_frames = []


for prediction_date in sorted(
    leakage_model_episodes[
        "prediction_date"
    ]
    .unique()
):

    prediction_date = pd.Timestamp(
        prediction_date
    )


    episode_population = (
        leakage_model_episodes.loc[
            leakage_model_episodes[
                "prediction_date"
            ]
            ==
            prediction_date
        ]
        .copy()
    )


    customers = (
        episode_population[
            "customer"
        ]
        .unique()
    )


    # ------------------------------------------------------------
    # Window boundaries
    # ------------------------------------------------------------

    baseline_start = (
        prediction_date
        -
        pd.Timedelta(
            days=90
        )
    )


    prior90_start = (
        prediction_date
        -
        pd.Timedelta(
            days=180
        )
    )


    recent30_start = (
        prediction_date
        -
        pd.Timedelta(
            days=30
        )
    )


    # ------------------------------------------------------------
    # Current 90 days
    # ------------------------------------------------------------

    b90 = summarise_customer_window(
        daily_data=customer_daily,
        customers=customers,
        start_date=baseline_start,
        end_date=prediction_date,
        prefix="b90"
    )


    # ------------------------------------------------------------
    # Previous 90 days
    # ------------------------------------------------------------

    p90 = summarise_customer_window(
        daily_data=customer_daily,
        customers=customers,
        start_date=prior90_start,
        end_date=baseline_start,
        prefix="p90"
    )


    # ------------------------------------------------------------
    # Most recent 30 days
    # ------------------------------------------------------------

    r30 = summarise_customer_window(
        daily_data=customer_daily,
        customers=customers,
        start_date=recent30_start,
        end_date=prediction_date,
        prefix="r30"
    )


    # ------------------------------------------------------------
    # Product breadth in current 90 days
    # ------------------------------------------------------------

    b90_products = summarise_product_window(
        line_data=positive_customer_purchases,
        customers=customers,
        start_date=baseline_start,
        end_date=prediction_date,
        prefix="b90"
    )


    # ------------------------------------------------------------
    # Product breadth in previous 90 days
    # ------------------------------------------------------------

    p90_products = summarise_product_window(
        line_data=positive_customer_purchases,
        customers=customers,
        start_date=prior90_start,
        end_date=baseline_start,
        prefix="p90"
    )


    # ------------------------------------------------------------
    # Current 90-day cadence
    # ------------------------------------------------------------

    b90_gaps = summarise_purchase_gaps(
        daily_data=customer_daily,
        customers=customers,
        start_date=baseline_start,
        end_date=prediction_date,
        prefix="b90"
    )


    # ------------------------------------------------------------
    # Merge
    # ------------------------------------------------------------

    features = (
        episode_population
        .merge(
            b90,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            p90,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            r30,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            b90_products,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            p90_products,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            b90_gaps,
            on="customer",
            how="left",
            validate="one_to_one"
        )
    )


    feature_frames.append(
        features
    )

In [ ]:
leakage_feature_table = pd.concat(
    feature_frames,
    ignore_index=True
)


print(
    f"Feature-table episodes: "
    f"{len(leakage_feature_table):,}"
)

In [ ]:
# ================================================================
# 40.9 — STRUCTURAL ZERO FILL
# ================================================================

zero_fill_features = [
    "p90_revenue",
    "p90_orders",
    "p90_active_days",
    "p90_units",
    "p90_unique_products",
    "r30_revenue",
    "r30_orders",
    "r30_active_days",
    "r30_units",
]


for feature in zero_fill_features:

    if feature in leakage_feature_table.columns:

        leakage_feature_table[
            feature
        ] = (
            leakage_feature_table[
                feature
            ]
            .fillna(0)
        )

In [ ]:
# ================================================================
# 40.10 — BASELINE RECONSTRUCTION CHECK
# ================================================================

baseline_difference = (
    leakage_feature_table[
        "baseline_revenue"
    ]
    -
    leakage_feature_table[
        "b90_revenue"
    ]
)


print(
    "Maximum absolute baseline revenue difference: "
    f"{baseline_difference.abs().max():.8f}"
)


assert np.allclose(
    leakage_feature_table[
        "baseline_revenue"
    ],
    leakage_feature_table[
        "b90_revenue"
    ]
)


print(
    "✓ Baseline revenue reconstruction confirmed."
)

In [ ]:
# ================================================================
# 40.11 — ECONOMIC FEATURES
# ================================================================

leakage_feature_table[
    "b90_average_order_value"
] = (
    leakage_feature_table[
        "b90_revenue"
    ]
    /
    leakage_feature_table[
        "b90_orders"
    ]
)


leakage_feature_table[
    "b90_revenue_per_active_day"
] = (
    leakage_feature_table[
        "b90_revenue"
    ]
    /
    leakage_feature_table[
        "b90_active_days"
    ]
)


leakage_feature_table[
    "b90_units_per_order"
] = (
    leakage_feature_table[
        "b90_units"
    ]
    /
    leakage_feature_table[
        "b90_orders"
    ]
)


leakage_feature_table[
    "b90_products_per_order"
] = (
    leakage_feature_table[
        "b90_unique_products"
    ]
    /
    leakage_feature_table[
        "b90_orders"
    ]
)

In [ ]:
# ================================================================
# 40.12 — PREVIOUS 90-DAY ECONOMICS
# ================================================================

leakage_feature_table[
    "p90_average_order_value"
] = np.where(
    leakage_feature_table[
        "p90_orders"
    ]
    > 0,

    leakage_feature_table[
        "p90_revenue"
    ]
    /
    leakage_feature_table[
        "p90_orders"
    ],

    np.nan
)

In [ ]:
# ================================================================
# 40.13 — REVENUE TREND FEATURES
# ================================================================

leakage_feature_table[
    "revenue_change_vs_previous_90d"
] = (
    leakage_feature_table[
        "b90_revenue"
    ]
    -
    leakage_feature_table[
        "p90_revenue"
    ]
)


leakage_feature_table[
    "revenue_ratio_vs_previous_90d"
] = np.where(
    leakage_feature_table[
        "p90_revenue"
    ]
    > 0,

    leakage_feature_table[
        "b90_revenue"
    ]
    /
    leakage_feature_table[
        "p90_revenue"
    ],

    np.nan
)


leakage_feature_table[
    "log_revenue_ratio_vs_previous_90d"
] = (
    np.log1p(
        leakage_feature_table[
            "b90_revenue"
        ]
    )
    -
    np.log1p(
        leakage_feature_table[
            "p90_revenue"
        ]
    )
)

In [ ]:
# ================================================================
# 40.14 — RECENT 30-DAY MOMENTUM
# ================================================================

leakage_feature_table[
    "recent30_revenue_share"
] = (
    leakage_feature_table[
        "r30_revenue"
    ]
    /
    leakage_feature_table[
        "b90_revenue"
    ]
)


leakage_feature_table[
    "recent30_order_share"
] = (
    leakage_feature_table[
        "r30_orders"
    ]
    /
    leakage_feature_table[
        "b90_orders"
    ]
)


earlier60_revenue = (
    leakage_feature_table[
        "b90_revenue"
    ]
    -
    leakage_feature_table[
        "r30_revenue"
    ]
)


leakage_feature_table[
    "recent30_vs_earlier60_revenue_rate"
] = np.where(
    earlier60_revenue
    > 0,

    (
        leakage_feature_table[
            "r30_revenue"
        ]
        / 30
    )
    /
    (
        earlier60_revenue
        / 60
    ),

    np.nan
)

In [ ]:
# ================================================================
# 40.15 — RECENCY
# ================================================================

leakage_feature_table[
    "days_since_last_purchase"
] = (
    leakage_feature_table[
        "prediction_date"
    ]
    -
    leakage_feature_table[
        "b90_last_purchase_day"
    ]
).dt.days

In [ ]:
# ================================================================
# 40.16 — FREQUENCY FEATURES
# ================================================================

leakage_feature_table[
    "b90_orders_per_30_days"
] = (
    leakage_feature_table[
        "b90_orders"
    ]
    /
    3
)


leakage_feature_table[
    "b90_active_day_rate"
] = (
    leakage_feature_table[
        "b90_active_days"
    ]
    /
    90
)


leakage_feature_table[
    "p90_active_day_rate"
] = (
    leakage_feature_table[
        "p90_active_days"
    ]
    /
    90
)

In [ ]:
# ================================================================
# 40.17 — PRODUCT BREADTH TREND
# ================================================================

leakage_feature_table[
    "product_breadth_change"
] = (
    leakage_feature_table[
        "b90_unique_products"
    ]
    -
    leakage_feature_table[
        "p90_unique_products"
    ]
)

In [ ]:
# ================================================================
# 40.18 — LOG-SCALE FEATURES
# ================================================================

leakage_feature_table[
    "log1p_b90_revenue"
] = np.log1p(
    leakage_feature_table[
        "b90_revenue"
    ]
)


leakage_feature_table[
    "log1p_p90_revenue"
] = np.log1p(
    leakage_feature_table[
        "p90_revenue"
    ]
)


leakage_feature_table[
    "log1p_b90_orders"
] = np.log1p(
    leakage_feature_table[
        "b90_orders"
    ]
)

In [ ]:
# ================================================================
# 40.19 — CYCLICAL CALENDAR FEATURES
# ================================================================

leakage_feature_table[
    "prediction_month"
] = (
    leakage_feature_table[
        "prediction_date"
    ]
    .dt.month
)


leakage_feature_table[
    "prediction_month_sin"
] = np.sin(
    2
    *
    np.pi
    *
    leakage_feature_table[
        "prediction_month"
    ]
    /
    12
)


leakage_feature_table[
    "prediction_month_cos"
] = np.cos(
    2
    *
    np.pi
    *
    leakage_feature_table[
        "prediction_month"
    ]
    /
    12
)

In [ ]:
# ================================================================
# 40.20 — SAFE PRIOR LEAKAGE HISTORY
# ================================================================

leakage_feature_table[
    "prior_completed_leakage_episodes"
] = 0


leakage_feature_table[
    "prior_leakage_events"
] = 0


leakage_feature_table[
    "prior_leakage_rate"
] = np.nan

In [ ]:
history_source = (
    revenue_leakage_episodes[
        [
            "customer",
            "prediction_date",
            "outcome_end",
            "revenue_leakage_event",
        ]
    ]
    .sort_values(
        [
            "customer",
            "prediction_date"
        ]
    )
)


for customer, current_group in (
    leakage_feature_table
    .groupby(
        "customer"
    )
):

    customer_history = (
        history_source.loc[
            history_source[
                "customer"
            ]
            ==
            customer
        ]
    )


    for index, row in (
        current_group
        .iterrows()
    ):

        known_prior = (
            customer_history.loc[
                (
                    customer_history[
                        "prediction_date"
                    ]
                    <
                    row[
                        "prediction_date"
                    ]
                )
                &
                (
                    customer_history[
                        "outcome_end"
                    ]
                    <=
                    row[
                        "prediction_date"
                    ]
                )
            ]
        )


        prior_count = len(
            known_prior
        )


        prior_events = (
            known_prior[
                "revenue_leakage_event"
            ]
            .sum()
        )


        leakage_feature_table.loc[
            index,
            "prior_completed_leakage_episodes"
        ] = prior_count


        leakage_feature_table.loc[
            index,
            "prior_leakage_events"
        ] = prior_events


        if prior_count > 0:

            leakage_feature_table.loc[
                index,
                "prior_leakage_rate"
            ] = (
                prior_events
                /
                prior_count
            )

In [ ]:
# ================================================================
# 40.21 — PRIOR LEAKAGE INDICATOR
# ================================================================

leakage_feature_table[
    "has_prior_completed_leakage_episode"
] = (
    leakage_feature_table[
        "prior_completed_leakage_episodes"
    ]
    > 0
).astype(int)

In [ ]:
# ================================================================
# 40.22 — FEATURE REGISTER
# ================================================================

leakage_numeric_features = [
    # Current 90-day economics
    "b90_revenue",
    "b90_orders",
    "b90_active_days",
    "b90_units",
    "b90_unique_products",
    "b90_average_unit_price",
    "b90_maximum_unit_price",
    "b90_average_order_value",
    "b90_revenue_per_active_day",
    "b90_units_per_order",
    "b90_products_per_order",

    # Previous 90-day behaviour
    "p90_revenue",
    "p90_orders",
    "p90_active_days",
    "p90_units",
    "p90_unique_products",
    "p90_average_order_value",

    # Recent 30 days
    "r30_revenue",
    "r30_orders",
    "r30_active_days",
    "r30_units",

    # Trends
    "revenue_change_vs_previous_90d",
    "revenue_ratio_vs_previous_90d",
    "log_revenue_ratio_vs_previous_90d",
    "recent30_revenue_share",
    "recent30_order_share",
    "recent30_vs_earlier60_revenue_rate",
    "product_breadth_change",

    # Recency and cadence
    "days_since_last_purchase",
    "b90_average_gap_days",
    "b90_maximum_gap_days",
    "b90_minimum_gap_days",
    "b90_orders_per_30_days",
    "b90_active_day_rate",
    "p90_active_day_rate",

    # Prior leakage history
    "prior_completed_leakage_episodes",
    "prior_leakage_events",
    "prior_leakage_rate",
    "has_prior_completed_leakage_episode",

    # Transforms
    "log1p_b90_revenue",
    "log1p_p90_revenue",
    "log1p_b90_orders",

    # Calendar
    "prediction_month_sin",
    "prediction_month_cos",
]


leakage_categorical_features = [
    "last_region",
]


leakage_model_features = (
    leakage_numeric_features
    +
    leakage_categorical_features
)

In [ ]:
# ================================================================
# 40.23 — LEAKAGE AUDIT
# ================================================================

forbidden_leakage_features = [
    "future_revenue",
    "future_orders",
    "future_active_days",
    "future_to_baseline_revenue_ratio",
    "revenue_leakage_amount",
    "future_inactive",
    "material_contraction",
    "revenue_leakage_event",
]


feature_overlap = (
    set(
        leakage_model_features
    )
    &
    set(
        forbidden_leakage_features
    )
)


assert not feature_overlap, (
    f"Target leakage detected: "
    f"{feature_overlap}"
)


print(
    "✓ No future outcome variables in feature register."
)

In [ ]:
# ================================================================
# 40.24 — FINAL STEP 40 MODELLING TABLE
# ================================================================

leakage_model_dataset = (
    leakage_feature_table[
        [
            "customer",
            "prediction_date",
            "prediction_period",
            "baseline_revenue",
            "revenue_leakage_event",

            *leakage_model_features,
        ]
    ]
    .copy()
)


leakage_model_dataset = (
    leakage_model_dataset
    .sort_values(
        [
            "prediction_date",
            "customer"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Model episodes: "
    f"{len(leakage_model_dataset):,}"
)

print(
    f"Unique customers: "
    f"{leakage_model_dataset['customer'].nunique():,}"
)

print(
    f"Overall target prevalence: "
    f"{leakage_model_dataset['revenue_leakage_event'].mean() * 100:.2f}%"
)

In [ ]:
# ================================================================
# 40.25 — FEATURE MISSINGNESS
# ================================================================

leakage_feature_missingness = (
    leakage_model_dataset[
        leakage_model_features
    ]
    .isna()
    .mean()
    .mul(100)
    .rename(
        "missing_pct"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature"
        }
    )
    .sort_values(
        "missing_pct",
        ascending=False
    )
)


display(
    leakage_feature_missingness
)

### 40.26 — Primary temporal split
Because this is a 90-day forward target, use deliberately separated periods.

We will use:
- Train: May 2010 → November 2010
- Embargo: December 2010 → February 2011
- Validation: March 2011 → April 2011
- Embargo: May 2011 → June 2011
- Test: July 2011 → August 2011

This is more conservative than the earlier notebooks.

In [ ]:
# ================================================================
# 40.26 — PURGED TEMPORAL SPLIT
# ================================================================

TRAIN_START = pd.Period(
    "2010-05",
    freq="M"
)

TRAIN_END = pd.Period(
    "2010-11",
    freq="M"
)


VALIDATION_START = pd.Period(
    "2011-03",
    freq="M"
)

VALIDATION_END = pd.Period(
    "2011-04",
    freq="M"
)


TEST_START = pd.Period(
    "2011-07",
    freq="M"
)

TEST_END = pd.Period(
    "2011-08",
    freq="M"
)

### 40.27 — Create Train / Validation / Test

In [ ]:
# ================================================================
# 40.27 — CREATE SPLITS
# ================================================================

leakage_train_data = (
    leakage_model_dataset.loc[
        leakage_model_dataset[
            "prediction_period"
        ]
        .between(
            TRAIN_START,
            TRAIN_END
        )
    ]
    .copy()
)


leakage_validation_data = (
    leakage_model_dataset.loc[
        leakage_model_dataset[
            "prediction_period"
        ]
        .between(
            VALIDATION_START,
            VALIDATION_END
        )
    ]
    .copy()
)


leakage_test_data = (
    leakage_model_dataset.loc[
        leakage_model_dataset[
            "prediction_period"
        ]
        .between(
            TEST_START,
            TEST_END
        )
    ]
    .copy()
)

In [ ]:
# ================================================================
# 40.28 — EMBARGO POPULATIONS
# ================================================================

embargo_1_data = (
    leakage_model_dataset.loc[
        leakage_model_dataset[
            "prediction_period"
        ]
        .between(
            pd.Period(
                "2010-12",
                freq="M"
            ),
            pd.Period(
                "2011-02",
                freq="M"
            )
        )
    ]
)


embargo_2_data = (
    leakage_model_dataset.loc[
        leakage_model_dataset[
            "prediction_period"
        ]
        .between(
            pd.Period(
                "2011-05",
                freq="M"
            ),
            pd.Period(
                "2011-06",
                freq="M"
            )
        )
    ]
)


print(
    f"Embargo 1 episodes: "
    f"{len(embargo_1_data):,}"
)

print(
    f"Embargo 2 episodes: "
    f"{len(embargo_2_data):,}"
)

In [ ]:
# ================================================================
# 40.29 — DEVELOPMENT SPLIT SUMMARY
# ================================================================

leakage_development_split_summary = pd.DataFrame(
    [
        {
            "Dataset":
                "Train",

            "Episodes":
                len(
                    leakage_train_data
                ),

            "Unique Customers":
                leakage_train_data[
                    "customer"
                ].nunique(),

            "Start":
                leakage_train_data[
                    "prediction_period"
                ].min(),

            "End":
                leakage_train_data[
                    "prediction_period"
                ].max(),

            "Leakage Rate":
                leakage_train_data[
                    "revenue_leakage_event"
                ].mean(),
        },

        {
            "Dataset":
                "Validation",

            "Episodes":
                len(
                    leakage_validation_data
                ),

            "Unique Customers":
                leakage_validation_data[
                    "customer"
                ].nunique(),

            "Start":
                leakage_validation_data[
                    "prediction_period"
                ].min(),

            "End":
                leakage_validation_data[
                    "prediction_period"
                ].max(),

            "Leakage Rate":
                leakage_validation_data[
                    "revenue_leakage_event"
                ].mean(),
        },
    ]
)


leakage_development_split_summary[
    "Leakage Rate (%)"
] = (
    leakage_development_split_summary[
        "Leakage Rate"
    ]
    *
    100
)


display(
    leakage_development_split_summary
)

In [ ]:
# ================================================================
# 40.30 — TEST GOVERNANCE
# ================================================================

print(
    f"Test episodes reserved: "
    f"{len(leakage_test_data):,}"
)

print(
    f"Test period: "
    f"{leakage_test_data['prediction_period'].min()} "
    f"to "
    f"{leakage_test_data['prediction_period'].max()}"
)


LEAKAGE_TEST_SET_OPENED = False


print(
    "Test target prevalence remains intentionally uninspected."
)

In [ ]:
# ================================================================
# 40.31 — TRAIN / VALIDATION MATRICES
# ================================================================

X_leakage_train = (
    leakage_train_data[
        leakage_model_features
    ]
    .copy()
)


y_leakage_train = (
    leakage_train_data[
        "revenue_leakage_event"
    ]
    .copy()
)


X_leakage_validation = (
    leakage_validation_data[
        leakage_model_features
    ]
    .copy()
)


y_leakage_validation = (
    leakage_validation_data[
        "revenue_leakage_event"
    ]
    .copy()
)

In [ ]:
X_leakage_test = (
    leakage_test_data[
        leakage_model_features
    ]
    .copy()
)


y_leakage_test = (
    leakage_test_data[
        "revenue_leakage_event"
    ]
    .copy()
)

In [ ]:
# ================================================================
# 40.32 — TRAIN → VALIDATION NUMERIC DRIFT
# ================================================================

leakage_drift_records = []


for feature in leakage_numeric_features:

    train_values = pd.to_numeric(
        X_leakage_train[
            feature
        ],
        errors="coerce"
    )


    validation_values = pd.to_numeric(
        X_leakage_validation[
            feature
        ],
        errors="coerce"
    )


    train_mean = (
        train_values.mean()
    )


    validation_mean = (
        validation_values.mean()
    )


    pooled_std = np.sqrt(
        (
            train_values.std() ** 2
            +
            validation_values.std() ** 2
        )
        /
        2
    )


    if (
        pd.isna(
            pooled_std
        )
        or
        pooled_std == 0
    ):

        smd = np.nan

    else:

        smd = (
            validation_mean
            -
            train_mean
        ) / pooled_std


    leakage_drift_records.append(
        {
            "feature":
                feature,

            "train_mean":
                train_mean,

            "validation_mean":
                validation_mean,

            "SMD":
                smd,

            "absolute_SMD":
                abs(
                    smd
                )
                if not pd.isna(
                    smd
                )
                else np.nan,
        }
    )


leakage_feature_drift = (
    pd.DataFrame(
        leakage_drift_records
    )
    .sort_values(
        "absolute_SMD",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    leakage_feature_drift
)

In [ ]:
plt.figure(
    figsize=(11, 11)
)


sns.barplot(
    data=(
        leakage_feature_drift
        .sort_values(
            "absolute_SMD"
        )
    ),
    y="feature",
    x="absolute_SMD"
)


plt.axvline(
    0.10,
    linestyle="--",
    label="0.10"
)


plt.axvline(
    0.25,
    linestyle="--",
    label="0.25"
)


plt.title(
    "Revenue Leakage Model — Train-to-Validation Feature Drift"
)

plt.xlabel(
    "Absolute Standardised Mean Difference"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 40.34 — STEP 40 COMPLETION CHECK
# ================================================================

assert (
    len(
        leakage_train_data
    )
    > 0
)


assert (
    len(
        leakage_validation_data
    )
    > 0
)


assert (
    len(
        leakage_test_data
    )
    > 0
)


assert (
    set(
        leakage_train_data[
            "prediction_period"
        ]
    )
    .isdisjoint(
        set(
            leakage_validation_data[
                "prediction_period"
            ]
        )
    )
)


assert (
    set(
        leakage_validation_data[
            "prediction_period"
        ]
    )
    .isdisjoint(
        set(
            leakage_test_data[
                "prediction_period"
            ]
        )
    )
)


assert (
    LEAKAGE_TEST_SET_OPENED
    is False
)


print(
    "✓ Step 40 complete."
)

print(
    "✓ Two complete historical 90-day windows required."
)

print(
    "✓ Revenue, frequency, recency and momentum features created."
)

print(
    "✓ Prior leakage history constructed using only matured outcomes."
)

print(
    "✓ Calendar seasonality encoded cyclically."
)

print(
    "✓ Purged chronological Train / Validation / Test populations created."
)

print(
    "✓ Final Test target remains unopened."
)

### Step 40 Summary — Leakage-Safe Feature Engineering and Purged Temporal Split

Step 40 created the leakage-safe modelling dataset using only information available on or before each customer prediction date.

### Historical Coverage

Of the original prediction episodes, **33,367** had at least 180 days of preceding observation history.

Only **1,954 episodes** lacked a complete two-window historical lookback and were excluded from the primary modelling population.

The modelling dataset therefore retains the large majority of eligible customer episodes while allowing comparison between:

- the current 90-day baseline;
- and the immediately preceding 90-day period.

### Feature Reconstruction Was Exact

The reconstructed baseline revenue matched the target-construction baseline exactly.

The maximum absolute difference was:

> **0.00000000**

This confirms that the feature and target windows are aligned correctly.

### Missingness Is Primarily Structural

The largest missingness occurs in variables that require sufficient customer history.

Approximately:

- **61.4%** of episodes lack within-baseline purchase-gap statistics;
- **45.1%** lack previous-period average order value and direct revenue-ratio measures;
- **36.1%** lack a prior leakage rate;
- **21.6%** lack the recent-30-day versus earlier-60-day revenue-rate comparison.

These observations should not be removed.

The missingness itself reflects customer purchasing structure and will be handled through model preprocessing and missingness indicators.

### Purged Temporal Validation Population

The primary Train population contains:

- **15,337 episodes**
- **3,949 unique customers**
- **54.45% leakage prevalence**

The Validation population contains:

- **3,663 episodes**
- **2,135 unique customers**
- **50.07% leakage prevalence**

A future Test population of **3,938 episodes** has been reserved across July–August 2011.

Its target prevalence remains unopened.

Deliberate temporal embargo periods separate Train, Validation and Test.

### Core Customer Behaviour Is Relatively Stable

Most central behavioural variables show limited Train-to-Validation drift.

Examples include:

- baseline 90-day revenue;
- baseline orders;
- baseline active days;
- baseline units;
- customer recency;
- recent 30-day revenue;
- and baseline product breadth.

This indicates that the core customer-economics feature architecture generalises reasonably well across the development periods.

### Calendar Position Shows Extreme Structural Drift

The cyclical prediction-month feature shows very large distributional change because Train and Validation occupy different parts of the calendar year.

This is expected from the deliberately chronological split.

However, earlier analysis showed strong seasonal variation in leakage rates.

Calendar information may therefore contain genuine predictive value while also presenting a risk of historical-seasonality memorisation.

The modelling stage should explicitly compare:

> **models with calendar variables vs models without calendar variables**

rather than assuming calendar effects should automatically be retained.

### Prior Leakage History Accumulates Through Time

Prior-leakage history variables also show substantial temporal drift.

Examples include:

- prior completed leakage episodes;
- prior leakage-event count;
- prior leakage-history indicator;
- prior leakage rate.

This is economically coherent because later prediction dates naturally contain more completed historical episodes.

These features should therefore also be stress-tested rather than automatically trusted.

### Step 40 Modelling Position

The final feature architecture is suitable for modelling.

The next stage will establish:

- naive leakage baselines;
- simple recency and revenue-trend benchmarks;
- Logistic Regression;
- and diagnostic specifications excluding highly time-dependent feature families.

The Test population remains unopened.



## Step 41 — Baselines, Logistic Regression and Feature-Family Stress Tests

Step 41 establishes the first predictive benchmarks for customer revenue leakage.

The initial objective is to determine whether leakage can be predicted materially better than simple prevalence and behavioural rules.

The first model family is Logistic Regression because it provides:

- a transparent baseline;
- stable probability estimates;
- and a useful benchmark before non-linear models are introduced.

Five initial specifications will be compared:

1. Training Prevalence Baseline
2. Recency-Only Logistic Regression
3. Revenue-Momentum Logistic Regression
4. Full Logistic Regression
5. Diagnostic challengers excluding calendar or prior-leakage-history variables

The Test population remains unopened.

### Primary Metric

PR-AUC will be the primary selection metric.

Because leakage prevalence is approximately 50%, ROC-AUC, Brier score and commercial lift will also be central.

### Commercial Objective

The model should identify customers at elevated risk of future revenue deterioration.

However, probability alone will not determine final prioritisation.

A later stage will combine leakage probability with customer revenue exposure.

In [ ]:
# ================================================================
# 41.2 — MODELLING IMPORTS
# ================================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
)

import numpy as np
import pandas as pd

In [ ]:
# ================================================================
# 41.3 — TEST GOVERNANCE
# ================================================================

assert (
    LEAKAGE_TEST_SET_OPENED
    is False
)


print(
    "✓ Revenue Leakage Test population remains unopened."
)

print(
    "Step 41 uses Train + Validation only."
)

In [ ]:
# ================================================================
# 41.4 — DEVELOPMENT TARGET PREVALENCE
# ================================================================

leakage_prevalence_summary = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Validation",
        ],

        "Episodes": [
            len(
                y_leakage_train
            ),

            len(
                y_leakage_validation
            ),
        ],

        "Leakage Rate": [
            y_leakage_train.mean(),
            y_leakage_validation.mean(),
        ],
    }
)


leakage_prevalence_summary[
    "Leakage Rate (%)"
] = (
    leakage_prevalence_summary[
        "Leakage Rate"
    ]
    *
    100
)


display(
    leakage_prevalence_summary
)

In [ ]:
# ================================================================
# 41.5 — LIFT / CAPTURE FUNCTIONS
# ================================================================

def lift_at_fraction(
    y_true,
    y_probability,
    fraction=0.10
):

    y_true = np.asarray(
        y_true
    )

    y_probability = np.asarray(
        y_probability
    )


    if np.allclose(
        y_probability,
        y_probability[0]
    ):

        return 1.0


    evaluation = pd.DataFrame(
        {
            "actual":
                y_true,

            "probability":
                y_probability,
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(
            drop=True
        )
    )


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                *
                fraction
            )
        )
    )


    overall_rate = (
        evaluation[
            "actual"
        ]
        .mean()
    )


    top_rate = (
        evaluation
        .head(
            n_top
        )[
            "actual"
        ]
        .mean()
    )


    if overall_rate == 0:

        return np.nan


    return (
        top_rate
        /
        overall_rate
    )

In [ ]:
def capture_at_fraction(
    y_true,
    y_probability,
    fraction=0.10
):

    y_true = np.asarray(
        y_true
    )

    y_probability = np.asarray(
        y_probability
    )


    if np.allclose(
        y_probability,
        y_probability[0]
    ):

        return fraction


    evaluation = pd.DataFrame(
        {
            "actual":
                y_true,

            "probability":
                y_probability,
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(drop=True)
    )


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                *
                fraction
            )
        )
    )


    total_positives = (
        evaluation[
            "actual"
        ]
        .sum()
    )


    if total_positives == 0:

        return np.nan


    return (
        evaluation
        .head(
            n_top
        )[
            "actual"
        ]
        .sum()
        /
        total_positives
    )

In [ ]:
# ================================================================
# 41.6 — BINARY EVALUATION FUNCTION
# ================================================================

def evaluate_leakage_model(
    model_name,
    y_true,
    probabilities,
    threshold=0.50
):

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )


    predictions = (
        probabilities
        >=
        threshold
    ).astype(int)


    return pd.DataFrame(
        {
            "Model": [
                model_name
            ],

            "PR-AUC": [
                average_precision_score(
                    y_true,
                    probabilities
                )
            ],

            "ROC-AUC": [
                roc_auc_score(
                    y_true,
                    probabilities
                )
            ],

            "Brier Score": [
                brier_score_loss(
                    y_true,
                    probabilities
                )
            ],

            "Precision": [
                precision_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "Recall": [
                recall_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "F1": [
                f1_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "Lift @ 10%": [
                lift_at_fraction(
                    y_true,
                    probabilities,
                    0.10
                )
            ],

            "Lift @ 20%": [
                lift_at_fraction(
                    y_true,
                    probabilities,
                    0.20
                )
            ],

            "Capture @ 10%": [
                capture_at_fraction(
                    y_true,
                    probabilities,
                    0.10
                )
            ],

            "Capture @ 20%": [
                capture_at_fraction(
                    y_true,
                    probabilities,
                    0.20
                )
            ],
        }
    )

In [ ]:
# ================================================================
# 41.7 — TRAINING PREVALENCE BASELINE
# ================================================================

training_leakage_prevalence = (
    y_leakage_train.mean()
)


validation_prevalence_probabilities = np.full(
    len(
        y_leakage_validation
    ),
    training_leakage_prevalence
)


leakage_prevalence_baseline_results = (
    evaluate_leakage_model(
        model_name="Training Prevalence Baseline",
        y_true=y_leakage_validation,
        probabilities=validation_prevalence_probabilities,
    )
)


display(
    leakage_prevalence_baseline_results
)

### 41.8 — Recency-only baseline
Recency is likely one of our strongest simple predictors.

In [ ]:
# ================================================================
# 41.8 — RECENCY-ONLY LOGISTIC
# ================================================================

recency_features = [
    "days_since_last_purchase"
]


recency_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            recency_features,
        )
    ]
)


recency_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            recency_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
recency_logistic_model.fit(
    X_leakage_train[
        recency_features
    ],
    y_leakage_train
)

In [ ]:
validation_recency_probabilities = (
    recency_logistic_model
    .predict_proba(
        X_leakage_validation[
            recency_features
        ]
    )[:, 1]
)

In [ ]:
recency_logistic_results = (
    evaluate_leakage_model(
        model_name="Recency-Only Logistic",
        y_true=y_leakage_validation,
        probabilities=validation_recency_probabilities,
    )
)


display(
    recency_logistic_results
)

### 41.9 — Revenue-momentum benchmark
This benchmark asks whether recent revenue trajectory alone carries enough signal.

In [ ]:
# ================================================================
# 41.9 — REVENUE-MOMENTUM FEATURES
# ================================================================

revenue_momentum_features = [
    "log1p_b90_revenue",
    "log1p_p90_revenue",
    "log_revenue_ratio_vs_previous_90d",
    "recent30_revenue_share",
    "recent30_vs_earlier60_revenue_rate",
    "days_since_last_purchase",
]

In [ ]:
revenue_momentum_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            revenue_momentum_features,
        )
    ]
)

In [ ]:
revenue_momentum_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            revenue_momentum_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)


revenue_momentum_logistic_model.fit(
    X_leakage_train[
        revenue_momentum_features
    ],
    y_leakage_train
)

In [ ]:
validation_revenue_momentum_probabilities = (
    revenue_momentum_logistic_model
    .predict_proba(
        X_leakage_validation[
            revenue_momentum_features
        ]
    )[:, 1]
)


revenue_momentum_logistic_results = (
    evaluate_leakage_model(
        model_name="Revenue-Momentum Logistic",
        y_true=y_leakage_validation,
        probabilities=validation_revenue_momentum_probabilities,
    )
)


display(
    revenue_momentum_logistic_results
)

### 41.10 — Full preprocessing

In [ ]:
# ================================================================
# 41.10 — FULL LOGISTIC PREPROCESSOR
# ================================================================

leakage_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),
    ]
)


leakage_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        ),
    ]
)


leakage_full_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            leakage_numeric_transformer,
            leakage_numeric_features,
        ),

        (
            "categorical",
            leakage_categorical_transformer,
            leakage_categorical_features,
        ),
    ]
)

### 41.11 — Full Logistic Regression

In [ ]:
# ================================================================
# 41.11 — FULL LOGISTIC REGRESSION
# ================================================================

full_leakage_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            leakage_full_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


full_leakage_logistic_model.fit(
    X_leakage_train,
    y_leakage_train
)

In [ ]:
validation_full_leakage_probabilities = (
    full_leakage_logistic_model
    .predict_proba(
        X_leakage_validation
    )[:, 1]
)

In [ ]:
full_leakage_logistic_results = (
    evaluate_leakage_model(
        model_name="Full Logistic Regression",
        y_true=y_leakage_validation,
        probabilities=validation_full_leakage_probabilities,
    )
)


display(
    full_leakage_logistic_results
)

### 41.12 — No-calendar challenger
Remove:
- prediction_month_sin
- prediction_month_cos

In [ ]:
# ================================================================
# 41.12 — NO-CALENDAR FEATURE SET
# ================================================================

no_calendar_numeric_features = [
    feature
    for feature in leakage_numeric_features
    if feature not in [
        "prediction_month_sin",
        "prediction_month_cos",
    ]
]


no_calendar_features = (
    no_calendar_numeric_features
    +
    leakage_categorical_features
)

Preprocessor:

In [ ]:
no_calendar_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            no_calendar_numeric_features,
        ),

        (
            "categorical",
            leakage_categorical_transformer,
            leakage_categorical_features,
        ),
    ]
)

Model:

In [ ]:
no_calendar_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            no_calendar_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


no_calendar_logistic_model.fit(
    X_leakage_train[
        no_calendar_features
    ],
    y_leakage_train
)

Predict/evaluate:

In [ ]:
validation_no_calendar_probabilities = (
    no_calendar_logistic_model
    .predict_proba(
        X_leakage_validation[
            no_calendar_features
        ]
    )[:, 1]
)


no_calendar_logistic_results = (
    evaluate_leakage_model(
        model_name="No-Calendar Logistic",
        y_true=y_leakage_validation,
        probabilities=validation_no_calendar_probabilities,
    )
)


display(
    no_calendar_logistic_results
)

### 41.13 — No-prior-leakage-history challenger
Remove:

In [ ]:
# ================================================================
# 41.13 — NO PRIOR-LEAKAGE HISTORY
# ================================================================

prior_leakage_history_features = [
    "prior_completed_leakage_episodes",
    "prior_leakage_events",
    "prior_leakage_rate",
    "has_prior_completed_leakage_episode",
]


no_prior_history_numeric_features = [
    feature
    for feature in leakage_numeric_features
    if feature not in prior_leakage_history_features
]


no_prior_history_features = (
    no_prior_history_numeric_features
    +
    leakage_categorical_features
)

Preprocessor:

In [ ]:
no_prior_history_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            no_prior_history_numeric_features,
        ),

        (
            "categorical",
            leakage_categorical_transformer,
            leakage_categorical_features,
        ),
    ]
)

Train:

In [ ]:
no_prior_history_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            no_prior_history_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


no_prior_history_logistic_model.fit(
    X_leakage_train[
        no_prior_history_features
    ],
    y_leakage_train
)

Evaluate:

In [ ]:
validation_no_prior_history_probabilities = (
    no_prior_history_logistic_model
    .predict_proba(
        X_leakage_validation[
            no_prior_history_features
        ]
    )[:, 1]
)


no_prior_history_logistic_results = (
    evaluate_leakage_model(
        model_name="No-Prior-History Logistic",
        y_true=y_leakage_validation,
        probabilities=validation_no_prior_history_probabilities,
    )
)


display(
    no_prior_history_logistic_results
)

### 41.14 — Stable-core challenger
Now deliberately remove both strongly time-dependent families:
- calendar
- prior leakage history

In [ ]:
# ================================================================
# 41.14 — STABLE-CORE FEATURE SET
# ================================================================

stable_core_numeric_features = [
    feature
    for feature in leakage_numeric_features
    if feature not in (
        [
            "prediction_month_sin",
            "prediction_month_cos",
        ]
        +
        prior_leakage_history_features
    )
]


stable_core_features = (
    stable_core_numeric_features
    +
    leakage_categorical_features
)

In [ ]:
stable_core_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            stable_core_numeric_features,
        ),

        (
            "categorical",
            leakage_categorical_transformer,
            leakage_categorical_features,
        ),
    ]
)

In [ ]:
stable_core_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            stable_core_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


stable_core_logistic_model.fit(
    X_leakage_train[
        stable_core_features
    ],
    y_leakage_train
)

In [ ]:
validation_stable_core_probabilities = (
    stable_core_logistic_model
    .predict_proba(
        X_leakage_validation[
            stable_core_features
        ]
    )[:, 1]
)


stable_core_logistic_results = (
    evaluate_leakage_model(
        model_name="Stable-Core Logistic",
        y_true=y_leakage_validation,
        probabilities=validation_stable_core_probabilities,
    )
)


display(
    stable_core_logistic_results
)

In [ ]:
# ================================================================
# 41.15 — INITIAL MODEL COMPARISON
# ================================================================

step41_model_comparison = (
    pd.concat(
        [
            leakage_prevalence_baseline_results,
            recency_logistic_results,
            revenue_momentum_logistic_results,
            full_leakage_logistic_results,
            no_calendar_logistic_results,
            no_prior_history_logistic_results,
            stable_core_logistic_results,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 20%",
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    step41_model_comparison
)

In [ ]:
# ================================================================
# 41.16 — PR-AUC RELATIVE TO PREVALENCE
# ================================================================

validation_leakage_prevalence = (
    y_leakage_validation.mean()
)


step41_model_comparison[
    "PR-AUC / Prevalence"
] = (
    step41_model_comparison[
        "PR-AUC"
    ]
    /
    validation_leakage_prevalence
)


step41_model_comparison[
    "Normalised PR-AUC"
] = (
    (
        step41_model_comparison[
            "PR-AUC"
        ]
        -
        validation_leakage_prevalence
    )
    /
    (
        1
        -
        validation_leakage_prevalence
    )
)


display(
    step41_model_comparison
)

In [ ]:
# ================================================================
# 41.17 — DECILE PERFORMANCE
# ================================================================

def create_leakage_decile_performance(
    y_true,
    probabilities,
    n_deciles=10
):

    evaluation = pd.DataFrame(
        {
            "actual":
                np.asarray(
                    y_true
                ),

            "probability":
                np.asarray(
                    probabilities
                ),
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(drop=True)
    )


    evaluation[
        "score_decile"
    ] = (
        pd.qcut(
            np.arange(
                len(
                    evaluation
                )
            ),
            q=n_deciles,
            labels=False
        )
        + 1
    )


    overall_rate = (
        evaluation[
            "actual"
        ]
        .mean()
    )


    output = (
        evaluation
        .groupby(
            "score_decile",
            as_index=False
        )
        .agg(
            episodes=(
                "actual",
                "size"
            ),

            actual_leakage_rate=(
                "actual",
                "mean"
            ),

            average_predicted_probability=(
                "probability",
                "mean"
            ),

            leakage_events=(
                "actual",
                "sum"
            ),
        )
    )


    output[
        "actual_leakage_rate_pct"
    ] = (
        output[
            "actual_leakage_rate"
        ]
        *
        100
    )


    output[
        "average_predicted_probability_pct"
    ] = (
        output[
            "average_predicted_probability"
        ]
        *
        100
    )


    output[
        "lift"
    ] = (
        output[
            "actual_leakage_rate"
        ]
        /
        overall_rate
    )


    return output

In [ ]:
# ================================================================
# 41.18 — DECILES FOR EACH SERIOUS LOGISTIC CANDIDATE
# ================================================================

full_leakage_deciles = (
    create_leakage_decile_performance(
        y_leakage_validation,
        validation_full_leakage_probabilities
    )
)


no_calendar_leakage_deciles = (
    create_leakage_decile_performance(
        y_leakage_validation,
        validation_no_calendar_probabilities
    )
)


stable_core_leakage_deciles = (
    create_leakage_decile_performance(
        y_leakage_validation,
        validation_stable_core_probabilities
    )
)

In [ ]:
display(
    full_leakage_deciles
)

display(
    no_calendar_leakage_deciles
)

display(
    stable_core_leakage_deciles
)

In [ ]:
# ================================================================
# 41.19 — FULL LOGISTIC COEFFICIENTS
# ================================================================

full_leakage_preprocessor_fitted = (
    full_leakage_logistic_model
    .named_steps[
        "preprocessing"
    ]
)


full_leakage_classifier = (
    full_leakage_logistic_model
    .named_steps[
        "model"
    ]
)


full_leakage_feature_names = (
    full_leakage_preprocessor_fitted
    .get_feature_names_out()
)


full_leakage_coefficients = (
    full_leakage_classifier
    .coef_[0]
)


full_leakage_coefficient_table = pd.DataFrame(
    {
        "feature":
            full_leakage_feature_names,

        "coefficient":
            full_leakage_coefficients,
    }
)


full_leakage_coefficient_table[
    "odds_ratio"
] = np.exp(
    full_leakage_coefficient_table[
        "coefficient"
    ]
)


full_leakage_coefficient_table[
    "absolute_coefficient"
] = (
    full_leakage_coefficient_table[
        "coefficient"
    ]
    .abs()
)


full_leakage_coefficient_table[
    "clean_feature"
] = (
    full_leakage_coefficient_table[
        "feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)


full_leakage_coefficient_table = (
    full_leakage_coefficient_table
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    full_leakage_coefficient_table.head(30)
)

In [ ]:
# ================================================================
# 41.20 — LEAKAGE SEGMENT SUMMARY
# ================================================================

def leakage_segment_summary(
    y_true,
    probabilities,
    model_name
):

    scored = pd.DataFrame(
        {
            "actual":
                np.asarray(
                    y_true
                ),

            "score":
                np.asarray(
                    probabilities
                ),
        }
    )


    overall_rate = (
        scored[
            "actual"
        ]
        .mean()
    )


    segments = []


    for label, quantile, direction in [
        (
            "Top 10%",
            0.90,
            "top"
        ),
        (
            "Top 20%",
            0.80,
            "top"
        ),
        (
            "Bottom 20%",
            0.20,
            "bottom"
        ),
    ]:

        threshold = (
            scored[
                "score"
            ]
            .quantile(
                quantile
            )
        )


        if direction == "top":

            mask = (
                scored[
                    "score"
                ]
                >=
                threshold
            )

        else:

            mask = (
                scored[
                    "score"
                ]
                <=
                threshold
            )


        segment = (
            scored.loc[
                mask
            ]
        )


        observed_rate = (
            segment[
                "actual"
            ]
            .mean()
        )


        segments.append(
            {
                "Model":
                    model_name,

                "Segment":
                    label,

                "Episodes":
                    len(
                        segment
                    ),

                "Observed Leakage Rate":
                    observed_rate,

                "Observed Leakage Rate (%)":
                    observed_rate
                    *
                    100,

                "Lift vs Overall":
                    observed_rate
                    /
                    overall_rate,
            }
        )


    return pd.DataFrame(
        segments
    )

In [ ]:
# ================================================================
# 41.21 — SEGMENT COMPARISON
# ================================================================

step41_segment_comparison = pd.concat(
    [
        leakage_segment_summary(
            y_leakage_validation,
            validation_recency_probabilities,
            "Recency-Only Logistic"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_revenue_momentum_probabilities,
            "Revenue-Momentum Logistic"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_full_leakage_probabilities,
            "Full Logistic"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_no_calendar_probabilities,
            "No-Calendar Logistic"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_stable_core_probabilities,
            "Stable-Core Logistic"
        ),
    ],
    ignore_index=True
)


display(
    step41_segment_comparison
)

In [ ]:
plt.figure(
    figsize=(12, 7)
)


sns.barplot(
    data=step41_model_comparison,
    y="Model",
    x="PR-AUC"
)


plt.axvline(
    validation_leakage_prevalence,
    linestyle="--",
    label=(
        f"Validation prevalence "
        f"({validation_leakage_prevalence:.3f})"
    )
)


plt.title(
    "Revenue Leakage Models — Validation PR-AUC"
)

plt.xlabel(
    "PR-AUC"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(12, 7)
)


sns.barplot(
    data=step41_model_comparison,
    y="Model",
    x="Lift @ 20%"
)


plt.axvline(
    1,
    linestyle="--",
    label="Random targeting"
)


plt.title(
    "Revenue Leakage Models — Validation Lift@20%"
)

plt.xlabel(
    "Lift @ 20%"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 41.22 — STEP 41 COMPLETION CHECK
# ================================================================

assert (
    LEAKAGE_TEST_SET_OPENED
    is False
)


assert (
    len(
        validation_full_leakage_probabilities
    )
    ==
    len(
        y_leakage_validation
    )
)


assert (
    len(
        validation_stable_core_probabilities
    )
    ==
    len(
        y_leakage_validation
    )
)


print(
    "✓ Step 41 complete."
)

print(
    "✓ Naive prevalence baseline established."
)

print(
    "✓ Recency benchmark established."
)

print(
    "✓ Revenue-momentum benchmark established."
)

print(
    "✓ Full Logistic evaluated."
)

print(
    "✓ No-calendar stress test evaluated."
)

print(
    "✓ No-prior-history stress test evaluated."
)

print(
    "✓ Stable-core Logistic evaluated."
)

print(
    "✓ Test population remains unopened."
)

### Step 41 Summary — Initial Revenue Leakage Models

Step 41 established the first predictive benchmarks for future 90-day customer revenue leakage.

### Revenue Leakage Is Meaningfully Predictable

The strongest initial model, Full Logistic Regression, achieved approximately:

- **PR-AUC: 0.654**
- **ROC-AUC: 0.706**
- **Lift@10%: 1.34×**
- **Lift@20%: 1.37×**

The Validation leakage prevalence was approximately **50.1%**.

The model therefore provides substantial predictive information beyond simple prevalence.

The highest-scored 20% of customer episodes experienced approximately **68.6% leakage**, while the lowest-scored 20% experienced approximately **19.1% leakage**.

This represents commercially meaningful separation.

### Revenue Momentum Contains Strong Predictive Signal

A compact Revenue-Momentum Logistic model achieved approximately:

- **PR-AUC: 0.643**
- **ROC-AUC: 0.681**
- **Lift@10%: 1.36×**
- **Lift@20%: 1.32×**

This confirms that recent revenue trajectory, purchase recency and short-term activity provide a large proportion of the leakage-prediction signal.

### Stable-Core Features Remain Highly Competitive

The Stable-Core Logistic specification excluded both calendar variables and accumulated prior-leakage-history variables.

It still achieved approximately:

- **PR-AUC: 0.636**
- **ROC-AUC: 0.682**
- **Brier score: 0.224**
- **Lift@10%: 1.37×**

The highest Stable-Core score decile experienced approximately **68.4% leakage**, while the lowest experienced approximately **16.9%**.

The result demonstrates that substantial leakage signal exists in relatively stable customer-behaviour features.

### Calendar and Prior Leakage History Are Interdependent

Removing calendar variables alone caused substantial performance deterioration.

The No-Calendar Logistic achieved only approximately **0.531 PR-AUC**.

However, removing both calendar variables and prior-leakage-history variables restored performance to approximately **0.636 PR-AUC**.

This suggests that accumulated leakage-history features are strongly entangled with calendar time.

Because customers naturally accumulate more historical leakage episodes as the dataset progresses, these variables may partly encode dataset maturity rather than stable customer behaviour.

They should therefore be treated cautiously during subsequent model selection.

### Full Logistic Ranks Well but Is Less Well Calibrated

Full Logistic Regression produced the strongest discrimination but a Brier score of approximately **0.241**, compared with approximately **0.224** for Stable-Core Logistic.

The Full Logistic also produced very high predicted probabilities relative to observed leakage rates in several validation deciles.

This suggests that its ranking is stronger than its literal probability calibration.

Calibration will therefore require explicit evaluation before final model selection.

### Initial Model Position

Three feature architectures remain especially important:

1. **Full feature architecture** — strongest initial discrimination;
2. **Revenue-Momentum architecture** — compact behavioural benchmark;
3. **Stable-Core architecture** — strong performance with reduced temporal dependence.

The next stage will test whether non-linear models can improve ranking and whether any apparent improvement survives when highly time-dependent features are excluded.

The Test population remains unopened.


## Step 42 — Non-Linear Revenue Leakage Models

Step 42 evaluates whether non-linear models improve revenue-leakage prediction beyond Logistic Regression.

Two feature architectures are evaluated in parallel:

### Full Architecture

Includes:

- current behaviour;
- historical behaviour;
- revenue momentum;
- calendar seasonality;
- and prior leakage history.

### Stable-Core Architecture

Excludes:

- calendar features;
- accumulated prior-leakage-history features.

This comparison is important because several time-dependent variables exhibited substantial Train-to-Validation drift.

The objective is therefore not simply to identify the highest validation score.

The objective is to determine whether non-linear predictive improvement remains present after reducing dependence on potentially unstable temporal features.

The Test population remains unopened.

In [ ]:
# ================================================================
# 42.2 — NON-LINEAR MODEL IMPORTS
# ================================================================

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

In [ ]:
# ================================================================
# 42.3 — TEST GOVERNANCE
# ================================================================

assert LEAKAGE_TEST_SET_OPENED is False

print(
    "✓ Leakage Test population remains unopened."
)

In [ ]:
# ================================================================
# 42.4 — FULL TREE PREPROCESSOR
# ================================================================

full_tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    )
                ]
            ),
            leakage_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    ),
                ]
            ),
            leakage_categorical_features,
        ),
    ]
)

### 42.5 — Full Random Forest

In [ ]:
# ================================================================
# 42.5 — FULL RANDOM FOREST
# ================================================================

full_leakage_random_forest = Pipeline(
    steps=[
        (
            "preprocessing",
            full_tree_preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=10,
                min_samples_leaf=20,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)


full_leakage_random_forest.fit(
    X_leakage_train,
    y_leakage_train
)

In [ ]:
validation_full_rf_probabilities = (
    full_leakage_random_forest
    .predict_proba(
        X_leakage_validation
    )[:, 1]
)

In [ ]:
full_rf_results = (
    evaluate_leakage_model(
        model_name="Full Random Forest",
        y_true=y_leakage_validation,
        probabilities=validation_full_rf_probabilities,
    )
)


display(
    full_rf_results
)

### 42.6 — Full Gradient Boosting

In [ ]:
# ================================================================
# 42.6 — FULL GRADIENT BOOSTING
# ================================================================

full_leakage_gradient_boosting = Pipeline(
    steps=[
        (
            "preprocessing",
            full_tree_preprocessor
        ),

        (
            "model",
            GradientBoostingClassifier(
                n_estimators=250,
                learning_rate=0.03,
                max_depth=3,
                min_samples_leaf=20,
                random_state=42
            )
        ),
    ]
)


full_leakage_gradient_boosting.fit(
    X_leakage_train,
    y_leakage_train
)

In [ ]:
validation_full_gb_probabilities = (
    full_leakage_gradient_boosting
    .predict_proba(
        X_leakage_validation
    )[:, 1]
)


full_gb_results = (
    evaluate_leakage_model(
        model_name="Full Gradient Boosting",
        y_true=y_leakage_validation,
        probabilities=validation_full_gb_probabilities,
    )
)


display(
    full_gb_results
)

### 42.7 — Full HistGradientBoosting

In [ ]:
# ================================================================
# 42.7 — FULL HISTGRADIENTBOOSTING
# ================================================================

full_leakage_hist_gradient = Pipeline(
    steps=[
        (
            "preprocessing",
            full_tree_preprocessor
        ),

        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=30,
                l2_regularization=1.0,
                random_state=42
            )
        ),
    ]
)


full_leakage_hist_gradient.fit(
    X_leakage_train,
    y_leakage_train
)

In [ ]:
validation_full_hist_probabilities = (
    full_leakage_hist_gradient
    .predict_proba(
        X_leakage_validation
    )[:, 1]
)


full_hist_results = (
    evaluate_leakage_model(
        model_name="Full HistGradientBoosting",
        y_true=y_leakage_validation,
        probabilities=validation_full_hist_probabilities,
    )
)


display(
    full_hist_results
)

### 42.8 — Stable-Core tree preprocessor

In [ ]:
# ================================================================
# 42.8 — STABLE-CORE TREE PREPROCESSOR
# ================================================================

stable_tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    )
                ]
            ),
            stable_core_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    ),
                ]
            ),
            leakage_categorical_features,
        ),
    ]
)

### 42.9 — Stable-Core Random Forest

In [ ]:
# ================================================================
# 42.9 — STABLE-CORE RANDOM FOREST
# ================================================================

stable_core_random_forest = Pipeline(
    steps=[
        (
            "preprocessing",
            stable_tree_preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=10,
                min_samples_leaf=20,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)


stable_core_random_forest.fit(
    X_leakage_train[
        stable_core_features
    ],
    y_leakage_train
)

In [ ]:
validation_stable_rf_probabilities = (
    stable_core_random_forest
    .predict_proba(
        X_leakage_validation[
            stable_core_features
        ]
    )[:, 1]
)


stable_rf_results = (
    evaluate_leakage_model(
        model_name="Stable-Core Random Forest",
        y_true=y_leakage_validation,
        probabilities=validation_stable_rf_probabilities,
    )
)


display(
    stable_rf_results
)

### 42.10 — Stable-Core Gradient Boosting

In [ ]:
# ================================================================
# 42.10 — STABLE-CORE GRADIENT BOOSTING
# ================================================================

stable_core_gradient_boosting = Pipeline(
    steps=[
        (
            "preprocessing",
            stable_tree_preprocessor
        ),

        (
            "model",
            GradientBoostingClassifier(
                n_estimators=250,
                learning_rate=0.03,
                max_depth=3,
                min_samples_leaf=20,
                random_state=42
            )
        ),
    ]
)


stable_core_gradient_boosting.fit(
    X_leakage_train[
        stable_core_features
    ],
    y_leakage_train
)

In [ ]:
validation_stable_gb_probabilities = (
    stable_core_gradient_boosting
    .predict_proba(
        X_leakage_validation[
            stable_core_features
        ]
    )[:, 1]
)


stable_gb_results = (
    evaluate_leakage_model(
        model_name="Stable-Core Gradient Boosting",
        y_true=y_leakage_validation,
        probabilities=validation_stable_gb_probabilities,
    )
)


display(
    stable_gb_results
)

### 42.11 — Comprehensive comparison

In [ ]:
# ================================================================
# 42.11 — COMPLETE MODEL COMPARISON
# ================================================================

step42_model_comparison = (
    pd.concat(
        [
            leakage_prevalence_baseline_results,
            recency_logistic_results,
            revenue_momentum_logistic_results,
            stable_core_logistic_results,
            full_leakage_logistic_results,

            full_rf_results,
            full_gb_results,
            full_hist_results,

            stable_rf_results,
            stable_gb_results,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 20%",
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    step42_model_comparison
)

In [ ]:
# ================================================================
# 42.12 — PREVALENCE-ADJUSTED PERFORMANCE
# ================================================================

step42_model_comparison[
    "PR-AUC / Prevalence"
] = (
    step42_model_comparison[
        "PR-AUC"
    ]
    /
    validation_leakage_prevalence
)


step42_model_comparison[
    "Normalised PR-AUC"
] = (
    (
        step42_model_comparison[
            "PR-AUC"
        ]
        -
        validation_leakage_prevalence
    )
    /
    (
        1
        -
        validation_leakage_prevalence
    )
)


display(
    step42_model_comparison
)

In [ ]:
# ================================================================
# 42.13 — COMMERCIAL SEGMENT PERFORMANCE
# ================================================================

step42_segment_comparison = pd.concat(
    [
        leakage_segment_summary(
            y_leakage_validation,
            validation_full_leakage_probabilities,
            "Full Logistic"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_stable_core_probabilities,
            "Stable-Core Logistic"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_full_rf_probabilities,
            "Full Random Forest"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_full_gb_probabilities,
            "Full Gradient Boosting"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_full_hist_probabilities,
            "Full HistGradientBoosting"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_stable_rf_probabilities,
            "Stable-Core Random Forest"
        ),

        leakage_segment_summary(
            y_leakage_validation,
            validation_stable_gb_probabilities,
            "Stable-Core Gradient Boosting"
        ),
    ],
    ignore_index=True
)


display(
    step42_segment_comparison
)

In [ ]:
# ================================================================
# 42.14 — RANDOM FOREST DECILES
# ================================================================

full_rf_deciles = (
    create_leakage_decile_performance(
        y_leakage_validation,
        validation_full_rf_probabilities
    )
)


stable_rf_deciles = (
    create_leakage_decile_performance(
        y_leakage_validation,
        validation_stable_rf_probabilities
    )
)


display(
    full_rf_deciles
)

display(
    stable_rf_deciles
)

In [ ]:
# ================================================================
# 42.15 — FULL RANDOM FOREST FEATURE IMPORTANCE
# ================================================================

full_rf_preprocessor_fitted = (
    full_leakage_random_forest
    .named_steps[
        "preprocessing"
    ]
)


full_rf_classifier = (
    full_leakage_random_forest
    .named_steps[
        "model"
    ]
)


full_rf_feature_names = (
    full_rf_preprocessor_fitted
    .get_feature_names_out()
)


full_rf_importance = pd.DataFrame(
    {
        "feature":
            full_rf_feature_names,

        "importance":
            full_rf_classifier
            .feature_importances_,
    }
)


full_rf_importance[
    "clean_feature"
] = (
    full_rf_importance[
        "feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)


full_rf_importance = (
    full_rf_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    full_rf_importance.head(25)
)

In [ ]:
# ================================================================
# 42.16 — STABLE-CORE RF FEATURE IMPORTANCE
# ================================================================

stable_rf_preprocessor_fitted = (
    stable_core_random_forest
    .named_steps[
        "preprocessing"
    ]
)


stable_rf_classifier = (
    stable_core_random_forest
    .named_steps[
        "model"
    ]
)


stable_rf_feature_names = (
    stable_rf_preprocessor_fitted
    .get_feature_names_out()
)


stable_rf_importance = pd.DataFrame(
    {
        "feature":
            stable_rf_feature_names,

        "importance":
            stable_rf_classifier
            .feature_importances_,
    }
)


stable_rf_importance[
    "clean_feature"
] = (
    stable_rf_importance[
        "feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)


stable_rf_importance = (
    stable_rf_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    stable_rf_importance.head(25)
)

In [ ]:
# ================================================================
# 42.17 — RANKING CORRELATION
# ================================================================

leakage_model_score_table = pd.DataFrame(
    {
        "Full_Logistic":
            validation_full_leakage_probabilities,

        "Stable_Core_Logistic":
            validation_stable_core_probabilities,

        "Full_RF":
            validation_full_rf_probabilities,

        "Full_GB":
            validation_full_gb_probabilities,

        "Stable_RF":
            validation_stable_rf_probabilities,

        "Stable_GB":
            validation_stable_gb_probabilities,
    }
)


leakage_model_score_correlation = (
    leakage_model_score_table
    .corr(
        method="spearman"
    )
)


display(
    leakage_model_score_correlation.round(3)
)

In [ ]:
plt.figure(
    figsize=(12, 8)
)


sns.barplot(
    data=step42_model_comparison,
    y="Model",
    x="PR-AUC"
)


plt.axvline(
    validation_leakage_prevalence,
    linestyle="--",
    label="Validation prevalence"
)


plt.title(
    "Revenue Leakage Models — Validation PR-AUC"
)

plt.xlabel(
    "PR-AUC"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(12, 8)
)


sns.barplot(
    data=step42_model_comparison,
    y="Model",
    x="Lift @ 20%"
)


plt.axvline(
    1,
    linestyle="--"
)


plt.title(
    "Revenue Leakage Models — Validation Lift@20%"
)

plt.xlabel(
    "Lift @ 20%"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 42.18 — PRACTICAL EQUIVALENCE
# ================================================================

LEAKAGE_PR_AUC_TOLERANCE = 0.005


best_leakage_pr_auc = (
    step42_model_comparison[
        "PR-AUC"
    ]
    .max()
)


step42_model_comparison[
    "Within_PR_AUC_Equivalence_Band"
] = (
    step42_model_comparison[
        "PR-AUC"
    ]
    >=
    (
        best_leakage_pr_auc
        -
        LEAKAGE_PR_AUC_TOLERANCE
    )
)


display(
    step42_model_comparison
)

In [ ]:
# ================================================================
# 42.19 — STEP 42 COMPLETE
# ================================================================

assert (
    LEAKAGE_TEST_SET_OPENED
    is False
)


for probabilities in [
    validation_full_rf_probabilities,
    validation_full_gb_probabilities,
    validation_full_hist_probabilities,
    validation_stable_rf_probabilities,
    validation_stable_gb_probabilities,
]:

    assert (
        len(
            probabilities
        )
        ==
        len(
            y_leakage_validation
        )
    )


print(
    "✓ Step 42 complete."
)

print(
    "✓ Full non-linear models evaluated."
)

print(
    "✓ Stable-Core non-linear models evaluated."
)

print(
    "✓ Commercial ranking compared."
)

print(
    "✓ Feature importance prepared."
)

print(
    "✓ Test population remains unopened."
)

### Step 42 Summary — Non-Linear Revenue Leakage Models

Step 42 evaluated whether non-linear modelling materially improves customer revenue-leakage prediction beyond Logistic Regression.

### Three Models Are Practically Equivalent on PR-AUC

Using the predetermined PR-AUC practical-equivalence tolerance of 0.005, three models remain in contention:

1. **Full Logistic Regression — PR-AUC 0.654**
2. **Full Gradient Boosting — PR-AUC 0.654**
3. **Full HistGradientBoosting — PR-AUC 0.652**

The differences between these models are too small to justify selection on PR-AUC alone.

### Logistic Regression Provides the Strongest Broad Discrimination

Full Logistic Regression achieved the highest overall discrimination:

- **PR-AUC: 0.654**
- **ROC-AUC: 0.706**
- **Lift@10%: 1.34×**
- **Lift@20%: 1.37×**

Its lowest-ranked 20% of customer episodes experienced only approximately **19.1% leakage**, compared with approximately **50.1% leakage overall**.

This demonstrates strong separation across the full score distribution.

### Gradient Boosting Concentrates High-Risk Customers More Effectively

Full Gradient Boosting achieved approximately:

- **PR-AUC: 0.654**
- **ROC-AUC: 0.693**
- **Lift@10%: 1.42×**
- **Lift@20%: 1.40×**

The highest-ranked 10% of customers experienced approximately **71.1% leakage**.

Full HistGradientBoosting produced approximately **71.9% leakage** in its highest-risk decile.

The boosting models therefore provide stronger concentration at the highest-risk end of the ranking.

### Boosting Models Produce Better Probability Quality

The Full Logistic Regression Brier score was approximately **0.241**.

By comparison:

- Full Gradient Boosting: **0.222**
- Full HistGradientBoosting: **0.223**

This indicates that the boosting models provide more accurate probability estimates on the designated Validation population even though Logistic Regression retains slightly stronger broad discrimination.

### Random Forest Does Not Earn Champion Status

Full Random Forest achieved approximately **0.643 PR-AUC**, below the predetermined practical-equivalence range of the strongest model.

Its additional complexity therefore does not provide sufficient validation benefit to justify final-model contention.

### Stable-Core Models Confirm Genuine Behavioural Signal

Non-linear models built without calendar and accumulated prior-leakage-history variables still achieved approximately **0.64 PR-AUC** and strong top-decile lift.

This demonstrates that customer-specific revenue momentum, activity, recency and historical purchase behaviour contain substantial leakage signal independent of the most temporally sensitive feature families.

### Predictive Structure Is Economically Coherent

Random Forest feature importance was concentrated in:

- prior 90-day revenue;
- prior units and order activity;
- current-versus-prior revenue movement;
- product breadth;
- and calendar seasonality.

The model is therefore largely learning whether current customer behaviour represents deterioration relative to the customer's own recent economic history.

### Step 42 Model Position

Three primary contenders remain:

1. **Full Logistic Regression**
2. **Full Gradient Boosting**
3. **Full HistGradientBoosting**

The next stage will compare these models on:

- purged temporal robustness;
- calibration;
- bootstrap uncertainty;
- top-risk concentration;
- and stability relative to simpler behavioural specifications.

The Test population remains unopened.


## Step 43 — Temporal Robustness, Calibration and Champion Selection

Step 43 performs the final pre-test evaluation of the three strongest revenue-leakage models:

1. Full Logistic Regression;
2. Full Gradient Boosting;
3. Full HistGradientBoosting.

The models are practically equivalent on the primary PR-AUC metric but differ in:

- overall ranking quality;
- high-risk concentration;
- probability calibration;
- and model complexity.

This stage therefore evaluates whether their validation performance remains credible through historical time.

### Purged Historical Validation

Because every prediction requires a future 90-day target window, a training episode may only enter a historical fold if its target outcome was fully observable before the validation month began.

This prevents later-label information from entering the training population.

The final Test population remains unopened.

In [ ]:
# ================================================================
# 43.2 — TEST GOVERNANCE
# ================================================================

assert LEAKAGE_TEST_SET_OPENED is False

print(
    "✓ Revenue Leakage Test population remains unopened."
)

In [ ]:
# ================================================================
# 43.3 — TEMPORAL DIAGNOSTIC DATASET
# ================================================================

leakage_temporal_dataset = (
    leakage_feature_table[
        [
            "customer",
            "prediction_date",
            "prediction_period",
            "outcome_end",
            "revenue_leakage_event",
            *leakage_model_features,
        ]
    ]
    .sort_values(
        [
            "prediction_date",
            "customer"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Temporal diagnostic episodes: "
    f"{len(leakage_temporal_dataset):,}"
)

In [ ]:
# ================================================================
# 43.4 — HISTORICAL FOLD MONTHS
# ================================================================

PURGED_VALIDATION_PERIODS = [
    pd.Period(
        "2010-09",
        freq="M"
    ),
    pd.Period(
        "2010-10",
        freq="M"
    ),
    pd.Period(
        "2010-11",
        freq="M"
    ),
]


print(
    PURGED_VALIDATION_PERIODS
)

In [ ]:
# ================================================================
# 43.5 — PURGED TEMPORAL FOLDS
# ================================================================

purged_leakage_folds = []


for validation_period in PURGED_VALIDATION_PERIODS:

    validation_month_start = (
        validation_period
        .to_timestamp(
            how="start"
        )
    )


    validation_mask = (
        leakage_temporal_dataset[
            "prediction_period"
        ]
        ==
        validation_period
    )


    train_mask = (
        leakage_temporal_dataset[
            "outcome_end"
        ]
        <
        validation_month_start
    )


    fold_train = (
        leakage_temporal_dataset.loc[
            train_mask
        ]
        .copy()
    )


    fold_validation = (
        leakage_temporal_dataset.loc[
            validation_mask
        ]
        .copy()
    )


    if (
        len(
            fold_train
        )
        > 0
        and
        len(
            fold_validation
        )
        > 0
    ):

        purged_leakage_folds.append(
            {
                "validation_period":
                    validation_period,

                "train":
                    fold_train,

                "validation":
                    fold_validation,
            }
        )


print(
    f"Purged folds created: "
    f"{len(purged_leakage_folds)}"
)

In [ ]:
# ================================================================
# 43.6 — PURGED FOLD AUDIT
# ================================================================

purged_fold_audit = []


for fold in purged_leakage_folds:

    fold_train = fold[
        "train"
    ]

    fold_validation = fold[
        "validation"
    ]


    purged_fold_audit.append(
        {
            "validation_period":
                str(
                    fold[
                        "validation_period"
                    ]
                ),

            "train_episodes":
                len(
                    fold_train
                ),

            "validation_episodes":
                len(
                    fold_validation
                ),

            "latest_training_prediction":
                fold_train[
                    "prediction_date"
                ]
                .max(),

            "latest_training_outcome_known":
                fold_train[
                    "outcome_end"
                ]
                .max(),

            "validation_prediction_start":
                fold_validation[
                    "prediction_date"
                ]
                .min(),

            "validation_prevalence":
                fold_validation[
                    "revenue_leakage_event"
                ]
                .mean(),
        }
    )


purged_fold_audit = pd.DataFrame(
    purged_fold_audit
)


display(
    purged_fold_audit
)

In [ ]:
# ================================================================
# 43.7 — CLONE
# ================================================================

from sklearn.base import clone

In [ ]:
# ================================================================
# 43.8 — PURGED TEMPORAL MODEL EVALUATION
# ================================================================

purged_temporal_records = []


candidate_models = {
    "Full Logistic Regression":
        full_leakage_logistic_model,

    "Full Gradient Boosting":
        full_leakage_gradient_boosting,

    "Full HistGradientBoosting":
        full_leakage_hist_gradient,
}


for fold_number, fold in enumerate(
    purged_leakage_folds,
    start=1
):

    fold_train = (
        fold[
            "train"
        ]
    )

    fold_validation = (
        fold[
            "validation"
        ]
    )


    X_fold_train = (
        fold_train[
            leakage_model_features
        ]
    )


    y_fold_train = (
        fold_train[
            "revenue_leakage_event"
        ]
    )


    X_fold_validation = (
        fold_validation[
            leakage_model_features
        ]
    )


    y_fold_validation = (
        fold_validation[
            "revenue_leakage_event"
        ]
    )


    if (
        y_fold_validation.nunique()
        < 2
    ):

        continue


    for (
        model_name,
        base_model
    ) in candidate_models.items():

        fold_model = clone(
            base_model
        )


        fold_model.fit(
            X_fold_train,
            y_fold_train
        )


        probabilities = (
            fold_model
            .predict_proba(
                X_fold_validation
            )[:, 1]
        )


        purged_temporal_records.append(
            {
                "fold":
                    fold_number,

                "validation_period":
                    str(
                        fold[
                            "validation_period"
                        ]
                    ),

                "model":
                    model_name,

                "train_episodes":
                    len(
                        fold_train
                    ),

                "validation_episodes":
                    len(
                        fold_validation
                    ),

                "validation_prevalence":
                    y_fold_validation.mean(),

                "PR_AUC":
                    average_precision_score(
                        y_fold_validation,
                        probabilities
                    ),

                "ROC_AUC":
                    roc_auc_score(
                        y_fold_validation,
                        probabilities
                    ),

                "Brier_Score":
                    brier_score_loss(
                        y_fold_validation,
                        probabilities
                    ),

                "Lift_10":
                    lift_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.10
                    ),

                "Lift_20":
                    lift_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.20
                    ),

                "Capture_20":
                    capture_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.20
                    ),
            }
        )


purged_temporal_results = pd.DataFrame(
    purged_temporal_records
)


display(
    purged_temporal_results
)

In [ ]:
# ================================================================
# 43.9 — TEMPORAL ROBUSTNESS SUMMARY
# ================================================================

purged_temporal_summary = (
    purged_temporal_results
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        folds=(
            "fold",
            "nunique"
        ),

        mean_PR_AUC=(
            "PR_AUC",
            "mean"
        ),

        std_PR_AUC=(
            "PR_AUC",
            "std"
        ),

        min_PR_AUC=(
            "PR_AUC",
            "min"
        ),

        mean_ROC_AUC=(
            "ROC_AUC",
            "mean"
        ),

        mean_Brier=(
            "Brier_Score",
            "mean"
        ),

        mean_Lift_10=(
            "Lift_10",
            "mean"
        ),

        min_Lift_10=(
            "Lift_10",
            "min"
        ),

        mean_Lift_20=(
            "Lift_20",
            "mean"
        ),

        min_Lift_20=(
            "Lift_20",
            "min"
        ),

        mean_Capture_20=(
            "Capture_20",
            "mean"
        ),
    )
)


display(
    purged_temporal_summary
)

In [ ]:
plt.figure(
    figsize=(11, 6)
)


sns.lineplot(
    data=purged_temporal_results,
    x="validation_period",
    y="PR_AUC",
    hue="model",
    marker="o"
)


plt.title(
    "Revenue Leakage — Purged Historical PR-AUC"
)

plt.xlabel(
    "Historical Validation Month"
)

plt.ylabel(
    "PR-AUC"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(11, 6)
)


sns.lineplot(
    data=purged_temporal_results,
    x="validation_period",
    y="Lift_20",
    hue="model",
    marker="o"
)


plt.axhline(
    1,
    linestyle="--"
)


plt.title(
    "Revenue Leakage — Purged Historical Lift@20%"
)

plt.xlabel(
    "Historical Validation Month"
)

plt.ylabel(
    "Lift @ 20%"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 43.10 — FOLD WIN COUNTS
# ================================================================

purged_pr_winners = (
    purged_temporal_results.loc[
        purged_temporal_results
        .groupby(
            "validation_period"
        )[
            "PR_AUC"
        ]
        .idxmax(),
        [
            "validation_period",
            "model",
            "PR_AUC",
        ]
    ]
)


purged_lift20_winners = (
    purged_temporal_results.loc[
        purged_temporal_results
        .groupby(
            "validation_period"
        )[
            "Lift_20"
        ]
        .idxmax(),
        [
            "validation_period",
            "model",
            "Lift_20",
        ]
    ]
)


print(
    "PR-AUC fold winners:"
)

display(
    purged_pr_winners
)


print(
    "Lift@20 fold winners:"
)

display(
    purged_lift20_winners
)

In [ ]:
# ================================================================
# 43.11 — CALIBRATION
# ================================================================

from sklearn.calibration import calibration_curve

In [ ]:
# ================================================================
# 43.12 — VALIDATION CALIBRATION CURVES
# ================================================================

calibration_candidates = [
    (
        "Full Logistic",
        validation_full_leakage_probabilities
    ),

    (
        "Full Gradient Boosting",
        validation_full_gb_probabilities
    ),

    (
        "Full HistGradientBoosting",
        validation_full_hist_probabilities
    ),
]


plt.figure(
    figsize=(9, 7)
)


for (
    model_name,
    probabilities
) in calibration_candidates:

    prob_true, prob_pred = (
        calibration_curve(
            y_true=y_leakage_validation,
            y_prob=probabilities,
            n_bins=10,
            strategy="quantile"
        )
    )


    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=model_name
    )


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)


plt.title(
    "Revenue Leakage — Validation Calibration"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Leakage Rate"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 43.13 — CALIBRATION ERROR FUNCTION
# ================================================================

def mean_absolute_calibration_error(
    y_true,
    probabilities,
    n_bins=10
):

    prob_true, prob_pred = (
        calibration_curve(
            y_true=y_true,
            y_prob=probabilities,
            n_bins=n_bins,
            strategy="quantile"
        )
    )


    return (
        np.mean(
            np.abs(
                prob_true
                -
                prob_pred
            )
        )
        *
        100
    )

In [ ]:
leakage_calibration_summary = pd.DataFrame(
    {
        "Model": [
            "Full Logistic Regression",
            "Full Gradient Boosting",
            "Full HistGradientBoosting",
        ],

        "Mean Absolute Calibration Error (pp)": [
            mean_absolute_calibration_error(
                y_leakage_validation,
                validation_full_leakage_probabilities
            ),

            mean_absolute_calibration_error(
                y_leakage_validation,
                validation_full_gb_probabilities
            ),

            mean_absolute_calibration_error(
                y_leakage_validation,
                validation_full_hist_probabilities
            ),
        ],
    }
)


display(
    leakage_calibration_summary
)

In [ ]:
# ================================================================
# 43.14 — PAIRED BOOTSTRAP
# ================================================================

def paired_leakage_bootstrap(
    y_true,
    probabilities_a,
    probabilities_b,
    n_bootstrap=2000,
    random_state=42
):

    y_true = np.asarray(
        y_true
    )

    probabilities_a = np.asarray(
        probabilities_a
    )

    probabilities_b = np.asarray(
        probabilities_b
    )


    rng = np.random.default_rng(
        random_state
    )


    n = len(
        y_true
    )


    records = []


    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )


        y_sample = (
            y_true[
                indices
            ]
        )


        if (
            np.unique(
                y_sample
            ).size
            < 2
        ):

            continue


        a = (
            probabilities_a[
                indices
            ]
        )

        b = (
            probabilities_b[
                indices
            ]
        )


        records.append(
            {
                "PR_AUC_Difference":
                    average_precision_score(
                        y_sample,
                        a
                    )
                    -
                    average_precision_score(
                        y_sample,
                        b
                    ),

                "ROC_AUC_Difference":
                    roc_auc_score(
                        y_sample,
                        a
                    )
                    -
                    roc_auc_score(
                        y_sample,
                        b
                    ),

                "Lift10_Difference":
                    lift_at_fraction(
                        y_sample,
                        a,
                        0.10
                    )
                    -
                    lift_at_fraction(
                        y_sample,
                        b,
                        0.10
                    ),

                "Lift20_Difference":
                    lift_at_fraction(
                        y_sample,
                        a,
                        0.20
                    )
                    -
                    lift_at_fraction(
                        y_sample,
                        b,
                        0.20
                    ),
            }
        )


    return pd.DataFrame(
        records
    )

### 43.15 — Gradient Boosting vs Logistic bootstrap

In [ ]:
# ================================================================
# 43.15 — GB VS LOGISTIC
# ================================================================

gb_vs_logistic_bootstrap = (
    paired_leakage_bootstrap(
        y_true=y_leakage_validation,
        probabilities_a=validation_full_gb_probabilities,
        probabilities_b=validation_full_leakage_probabilities,
        n_bootstrap=2000,
        random_state=42
    )
)

### 43.16 — HistGradient vs Logistic bootstrap

In [ ]:
# ================================================================
# 43.16 — HISTGB VS LOGISTIC
# ================================================================

hist_vs_logistic_bootstrap = (
    paired_leakage_bootstrap(
        y_true=y_leakage_validation,
        probabilities_a=validation_full_hist_probabilities,
        probabilities_b=validation_full_leakage_probabilities,
        n_bootstrap=2000,
        random_state=42
    )
)

In [ ]:
# ================================================================
# 43.17 — BOOTSTRAP SUMMARY
# ================================================================

def summarise_pairwise_bootstrap(
    bootstrap,
    comparison_name
):

    records = []


    for metric in [
        "PR_AUC_Difference",
        "ROC_AUC_Difference",
        "Lift10_Difference",
        "Lift20_Difference",
    ]:

        records.append(
            {
                "Comparison":
                    comparison_name,

                "Metric":
                    metric,

                "Mean Difference":
                    bootstrap[
                        metric
                    ]
                    .mean(),

                "CI 2.5%":
                    bootstrap[
                        metric
                    ]
                    .quantile(
                        0.025
                    ),

                "CI 97.5%":
                    bootstrap[
                        metric
                    ]
                    .quantile(
                        0.975
                    ),

                "Probability First Model Better":
                    (
                        bootstrap[
                            metric
                        ]
                        > 0
                    )
                    .mean(),
            }
        )


    return pd.DataFrame(
        records
    )

In [ ]:
leakage_bootstrap_comparison = pd.concat(
    [
        summarise_pairwise_bootstrap(
            gb_vs_logistic_bootstrap,
            "Gradient Boosting - Logistic"
        ),

        summarise_pairwise_bootstrap(
            hist_vs_logistic_bootstrap,
            "HistGradientBoosting - Logistic"
        ),
    ],
    ignore_index=True
)


display(
    leakage_bootstrap_comparison
)

In [ ]:
# ================================================================
# 43.18 — FINAL PRE-TEST SCORECARD
# ================================================================

final_leakage_pretest_scorecard = pd.DataFrame(
    [
        {
            "Model":
                "Full Logistic Regression",

            "Validation PR-AUC":
                full_leakage_logistic_results[
                    "PR-AUC"
                ].iloc[0],

            "Validation ROC-AUC":
                full_leakage_logistic_results[
                    "ROC-AUC"
                ].iloc[0],

            "Validation Brier":
                full_leakage_logistic_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                full_leakage_logistic_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                full_leakage_logistic_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Calibration Error pp":
                leakage_calibration_summary.loc[
                    leakage_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Full Logistic Regression",
                    "Mean Absolute Calibration Error (pp)"
                ].iloc[0],
        },

        {
            "Model":
                "Full Gradient Boosting",

            "Validation PR-AUC":
                full_gb_results[
                    "PR-AUC"
                ].iloc[0],

            "Validation ROC-AUC":
                full_gb_results[
                    "ROC-AUC"
                ].iloc[0],

            "Validation Brier":
                full_gb_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                full_gb_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                full_gb_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Calibration Error pp":
                leakage_calibration_summary.loc[
                    leakage_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Full Gradient Boosting",
                    "Mean Absolute Calibration Error (pp)"
                ].iloc[0],
        },

        {
            "Model":
                "Full HistGradientBoosting",

            "Validation PR-AUC":
                full_hist_results[
                    "PR-AUC"
                ].iloc[0],

            "Validation ROC-AUC":
                full_hist_results[
                    "ROC-AUC"
                ].iloc[0],

            "Validation Brier":
                full_hist_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                full_hist_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                full_hist_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Calibration Error pp":
                leakage_calibration_summary.loc[
                    leakage_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Full HistGradientBoosting",
                    "Mean Absolute Calibration Error (pp)"
                ].iloc[0],
        },
    ]
)


display(
    final_leakage_pretest_scorecard
)

In [ ]:
# ================================================================
# 43.19 — CHAMPION DECISION PENDING
# ================================================================

LEAKAGE_MODEL_FROZEN = False

FINAL_LEAKAGE_MODEL_NAME = None
FINAL_LEAKAGE_MODEL = None


print(
    "Final Revenue Leakage model remains unfrozen "
    "pending Step 43 evidence review."
)

In [ ]:
# ================================================================
# 43.20 — STEP 43 COMPLETION CHECK
# ================================================================

assert (
    LEAKAGE_TEST_SET_OPENED
    is False
)


assert (
    LEAKAGE_MODEL_FROZEN
    is False
)


assert (
    len(
        purged_temporal_results
    )
    > 0
)


assert (
    len(
        leakage_bootstrap_comparison
    )
    > 0
)


print(
    "✓ Step 43 diagnostics complete."
)

print(
    "✓ Purged historical temporal validation completed."
)

print(
    "✓ Calibration assessed."
)

print(
    "✓ Paired bootstrap comparisons completed."
)

print(
    "✓ Test population remains unopened."
)

### Step 43 Summary — Temporal Robustness, Calibration and Final Champion Selection

Step 43 compared the three strongest revenue-leakage models using purged historical validation, calibration diagnostics and paired bootstrap analysis.

The final Test population remained unopened.

### Purged Temporal Validation Confirmed Genuine Historical Testing

Historical validation folds were constructed so that every training target had fully matured before the later validation prediction date.

For example, the September 2010 validation fold used only training episodes whose 90-day outcome windows had completed before the September prediction point.

This prevents future target information from entering historical model training.

### Logistic and Gradient Boosting Remain Very Close on Ranking

Across the designated Validation population:

- Full Logistic Regression achieved **PR-AUC 0.654**
- Full Gradient Boosting achieved **PR-AUC 0.654**
- Full HistGradientBoosting achieved **PR-AUC 0.652**

All three remained inside the predetermined practical-equivalence band.

Across the three purged historical folds, average PR-AUC was approximately:

- **Full Logistic: 0.691**
- **Gradient Boosting: 0.687**
- **HistGradientBoosting: 0.673**

Logistic therefore retained a small historical ranking advantage.

However, Gradient Boosting achieved the highest PR-AUC in two of the three historical folds.

### Logistic Retains Stronger Broad ROC Discrimination

Full Logistic Regression achieved validation ROC-AUC of approximately **0.706**, compared with approximately **0.693** for Gradient Boosting.

Paired bootstrap analysis estimated the Gradient Boosting minus Logistic ROC-AUC difference at approximately **−0.014**, with the 95% confidence interval remaining below zero.

The Logistic ROC-AUC advantage therefore appears statistically credible.

### PR-AUC Performance Is Effectively Equivalent

The Gradient Boosting minus Logistic PR-AUC difference was approximately **−0.001**.

Its bootstrap confidence interval extended from approximately **−0.016 to +0.015**.

There is therefore no meaningful evidence that either model is superior on PR-AUC.

### Gradient Boosting Provides Stronger High-Risk Concentration

Gradient Boosting achieved approximately:

- **Lift@10%: 1.42×**
- **Lift@20%: 1.40×**

compared with approximately:

- **Lift@10%: 1.34×**
- **Lift@20%: 1.37×**

for Full Logistic Regression.

Bootstrap analysis indicated approximately a **94% probability** that Gradient Boosting produced stronger top-decile lift on the Validation population, although the 95% interval still crossed zero.

### Calibration Strongly Favours Boosting

Probability calibration was the largest difference between the finalists.

Mean absolute calibration error was approximately:

- **Full Logistic Regression: 15.97 percentage points**
- **Full Gradient Boosting: 4.03 percentage points**
- **Full HistGradientBoosting: 3.26 percentage points**

Brier score also strongly favoured the boosting models.

On the designated Validation population:

- Logistic Brier: **0.241**
- Gradient Boosting Brier: **0.222**

Across purged historical folds, mean Brier was approximately:

- Logistic: **0.319**
- Gradient Boosting: **0.238**

The Logistic model therefore ranks customers effectively but produces probability estimates that are materially less reliable across time.

### Final Model Decision

The final Revenue Leakage model selected for freezing is:

> **Full Gradient Boosting**

This decision does not imply that Gradient Boosting dominates every statistical metric.

Full Logistic Regression remains slightly stronger on broad ROC discrimination and several historical ranking measures.

Gradient Boosting is selected because it provides the strongest overall operating balance of:

- practically equivalent PR-AUC;
- stronger high-risk concentration;
- substantially better Brier score;
- substantially better calibration;
- and credible performance across purged historical periods.

Probability quality is particularly important because the final commercial architecture will combine:

> **probability of leakage × financial exposure**

to prioritise revenue at risk.

For this use case, a well-ranked but severely miscalibrated probability model is less desirable than a comparably ranked model with substantially more credible probability estimates.

The final Test population remains unopened.


## STEP 44 — FREEZE THE MODEL & FINAL UNTOUCHED TEST
We can now freeze the specification.
### 44.1 — Freeze Full Gradient Boosting

In [ ]:
# ================================================================
# 44.1 — FREEZE FINAL REVENUE LEAKAGE MODEL
# ================================================================

FINAL_LEAKAGE_MODEL_NAME = (
    "Full Gradient Boosting"
)


FINAL_LEAKAGE_MODEL = (
    full_leakage_gradient_boosting
)


FINAL_LEAKAGE_FEATURES = (
    leakage_model_features.copy()
)


LEAKAGE_MODEL_FROZEN = True


print(
    f"Frozen model: "
    f"{FINAL_LEAKAGE_MODEL_NAME}"
)

print(
    f"Frozen feature count: "
    f"{len(FINAL_LEAKAGE_FEATURES)}"
)

print(
    "✓ Revenue Leakage model specification frozen."
)

In [ ]:
# ================================================================
# 44.2 — FROZEN MODEL SPECIFICATION
# ================================================================

frozen_leakage_specification = pd.DataFrame(
    {
        "Component": [
            "Prediction problem",
            "Prediction unit",
            "Baseline window",
            "Prediction point",
            "Outcome window",
            "Primary target",
            "Model",
            "Estimators",
            "Learning rate",
            "Maximum tree depth",
            "Minimum samples per leaf",
            "Numeric preprocessing",
            "Missingness indicators",
            "Categorical preprocessing",
            "Calendar representation",
            "Calibration layer",
            "Primary selection metric",
            "Commercial metrics",
            "Validation design",
            "Test status",
        ],

        "Frozen Specification": [
            "Future customer revenue leakage",

            "One active customer at one month-end prediction point",

            "Previous 90 days",

            "Month-end",

            "Following 90 days",

            "Future revenue <= 50% of prior 90-day revenue",

            "Gradient Boosting Classifier",

            250,

            0.03,

            3,

            20,

            "Median imputation",

            "Yes",

            "Most-frequent imputation + one-hot encoding",

            "Sine/cosine month encoding",

            "None",

            "PR-AUC",

            "Lift@10%, Lift@20%, Capture@10%, Capture@20%",

            "Purged chronological validation",

            "UNOPENED",
        ],
    }
)


display(
    frozen_leakage_specification
)

In [ ]:
# ================================================================
# 44.3 — FREEZE VALIDATION BENCHMARK
# ================================================================

frozen_leakage_validation_benchmark = (
    full_gb_results
    .copy()
)


frozen_leakage_validation_benchmark[
    "Evaluation Population"
] = (
    "Validation"
)


display(
    frozen_leakage_validation_benchmark
)

In [ ]:
# ================================================================
# 44.4 — PRE-TEST GOVERNANCE LOCK
# ================================================================

assert LEAKAGE_MODEL_FROZEN is True
assert LEAKAGE_TEST_SET_OPENED is False

assert (
    FINAL_LEAKAGE_MODEL_NAME
    ==
    "Full Gradient Boosting"
)


print(
    "✓ Final algorithm frozen."
)

print(
    "✓ Feature architecture frozen."
)

print(
    "✓ Hyperparameters frozen."
)

print(
    "✓ Validation benchmark frozen."
)

print(
    "✓ Test population remains unopened."
)

print(
    "No further validation-driven model changes are permitted."
)

## Step 45 — Final Untouched Out-of-Time Test Evaluation

All Revenue Leakage model-development decisions are now complete.

The frozen Full Gradient Boosting specification will be refitted using all historical episodes whose target outcomes were fully known before the beginning of the Test period.

The Test population will then be opened exactly once.

No model features, hyperparameters, preprocessing decisions or calibration rules will be changed after Test performance becomes available.

The objective is to determine whether the selected leakage model genuinely generalises to future customer-revenue deterioration.

In [ ]:
# ================================================================
# 45.2 — TEST START DATE
# ================================================================

LEAKAGE_TEST_START_DATE = pd.Timestamp(
    "2011-07-01"
)


print(
    f"Official Test period begins: "
    f"{LEAKAGE_TEST_START_DATE.date()}"
)

In [ ]:
# ================================================================
# 45.3 — FINAL DEVELOPMENT POPULATION
# ================================================================

final_leakage_development_data = (
    leakage_feature_table.loc[
        leakage_feature_table[
            "outcome_end"
        ]
        <
        LEAKAGE_TEST_START_DATE
    ]
    .copy()
    .sort_values(
        [
            "prediction_date",
            "customer"
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Final development episodes: "
    f"{len(final_leakage_development_data):,}"
)

print(
    f"Development prediction period: "
    f"{final_leakage_development_data['prediction_period'].min()} "
    f"to "
    f"{final_leakage_development_data['prediction_period'].max()}"
)

print(
    f"Latest matured development outcome: "
    f"{final_leakage_development_data['outcome_end'].max()}"
)

In [ ]:
# ================================================================
# 45.4 — TEMPORAL SAFETY CHECK
# ================================================================

assert (
    final_leakage_development_data[
        "outcome_end"
    ]
    .max()
    <
    LEAKAGE_TEST_START_DATE
)


assert (
    final_leakage_development_data[
        "prediction_period"
    ]
    .max()
    <
    TEST_START
)


print(
    "✓ Every development label matured before Test began."
)

In [ ]:
# ================================================================
# 45.5 — DEVELOPMENT MATRICES
# ================================================================

X_final_leakage_development = (
    final_leakage_development_data[
        FINAL_LEAKAGE_FEATURES
    ]
    .copy()
)


y_final_leakage_development = (
    final_leakage_development_data[
        "revenue_leakage_event"
    ]
    .copy()
)


print(
    f"Final development leakage prevalence: "
    f"{y_final_leakage_development.mean() * 100:.2f}%"
)

In [ ]:
# ================================================================
# 45.6 — CLONE FROZEN SPECIFICATION
# ================================================================

final_leakage_refit_model = clone(
    FINAL_LEAKAGE_MODEL
)


print(
    "✓ Exact frozen Gradient Boosting specification cloned."
)

In [ ]:
# ================================================================
# 45.7 — FINAL DEVELOPMENT REFIT
# ================================================================

final_leakage_refit_model.fit(
    X_final_leakage_development,
    y_final_leakage_development
)


print(
    "✓ Frozen model refitted on all label-matured "
    "pre-Test development episodes."
)

In [ ]:
# ================================================================
# 45.8 — OPEN FINAL TEST POPULATION
# ================================================================

LEAKAGE_TEST_SET_OPENED = True


print(
    "FINAL REVENUE LEAKAGE TEST POPULATION IS NOW OPEN."
)

print(
    "No further model changes are permitted."
)

In [ ]:
# ================================================================
# 45.9 — TEST TARGET PREVALENCE
# ================================================================

test_leakage_prevalence = (
    y_leakage_test.mean()
)


print(
    f"Final Test leakage prevalence: "
    f"{test_leakage_prevalence * 100:.2f}%"
)

In [ ]:
# ================================================================
# 45.10 — FINAL TEST PREDICTIONS
# ================================================================

final_leakage_test_probabilities = (
    final_leakage_refit_model
    .predict_proba(
        X_leakage_test[
            FINAL_LEAKAGE_FEATURES
        ]
    )[:, 1]
)


assert (
    len(
        final_leakage_test_probabilities
    )
    ==
    len(
        y_leakage_test
    )
)


print(
    "✓ Final Test probabilities generated."
)

In [ ]:
# ================================================================
# 45.11 — FINAL TEST METRICS
# ================================================================

final_leakage_test_results = (
    evaluate_leakage_model(
        model_name="Frozen Gradient Boosting — Final Test",
        y_true=y_leakage_test,
        probabilities=final_leakage_test_probabilities,
    )
)


display(
    final_leakage_test_results
)

In [ ]:
# ================================================================
# 45.12 — VALIDATION VS TEST
# ================================================================

leakage_validation_vs_test = pd.DataFrame(
    {
        "Metric": [
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ],

        "Validation": [
            frozen_leakage_validation_benchmark[
                "PR-AUC"
            ].iloc[0],

            frozen_leakage_validation_benchmark[
                "ROC-AUC"
            ].iloc[0],

            frozen_leakage_validation_benchmark[
                "Brier Score"
            ].iloc[0],

            frozen_leakage_validation_benchmark[
                "Lift @ 10%"
            ].iloc[0],

            frozen_leakage_validation_benchmark[
                "Lift @ 20%"
            ].iloc[0],

            frozen_leakage_validation_benchmark[
                "Capture @ 10%"
            ].iloc[0],

            frozen_leakage_validation_benchmark[
                "Capture @ 20%"
            ].iloc[0],
        ],

        "Test": [
            final_leakage_test_results[
                "PR-AUC"
            ].iloc[0],

            final_leakage_test_results[
                "ROC-AUC"
            ].iloc[0],

            final_leakage_test_results[
                "Brier Score"
            ].iloc[0],

            final_leakage_test_results[
                "Lift @ 10%"
            ].iloc[0],

            final_leakage_test_results[
                "Lift @ 20%"
            ].iloc[0],

            final_leakage_test_results[
                "Capture @ 10%"
            ].iloc[0],

            final_leakage_test_results[
                "Capture @ 20%"
            ].iloc[0],
        ],
    }
)


leakage_validation_vs_test[
    "Absolute Change"
] = (
    leakage_validation_vs_test[
        "Test"
    ]
    -
    leakage_validation_vs_test[
        "Validation"
    ]
)


display(
    leakage_validation_vs_test
)

In [ ]:
# ================================================================
# 45.13 — PREVALENCE-AWARE PR-AUC
# ================================================================

leakage_prevalence_adjusted_performance = pd.DataFrame(
    {
        "Dataset": [
            "Validation",
            "Test",
        ],

        "Prevalence": [
            validation_leakage_prevalence,
            test_leakage_prevalence,
        ],

        "PR-AUC": [
            frozen_leakage_validation_benchmark[
                "PR-AUC"
            ].iloc[0],

            final_leakage_test_results[
                "PR-AUC"
            ].iloc[0],
        ],
    }
)


leakage_prevalence_adjusted_performance[
    "PR-AUC / Prevalence"
] = (
    leakage_prevalence_adjusted_performance[
        "PR-AUC"
    ]
    /
    leakage_prevalence_adjusted_performance[
        "Prevalence"
    ]
)


leakage_prevalence_adjusted_performance[
    "Normalised PR-AUC"
] = (
    (
        leakage_prevalence_adjusted_performance[
            "PR-AUC"
        ]
        -
        leakage_prevalence_adjusted_performance[
            "Prevalence"
        ]
    )
    /
    (
        1
        -
        leakage_prevalence_adjusted_performance[
            "Prevalence"
        ]
    )
)


display(
    leakage_prevalence_adjusted_performance
)

In [ ]:
# ================================================================
# 45.14 — FINAL TEST DECILES
# ================================================================

final_leakage_test_deciles = (
    create_leakage_decile_performance(
        y_true=y_leakage_test,
        probabilities=final_leakage_test_probabilities
    )
)


display(
    final_leakage_test_deciles
)

In [ ]:
# ================================================================
# 45.15 — FINAL TEST CALIBRATION
# ================================================================

final_leakage_test_calibration_error = (
    mean_absolute_calibration_error(
        y_leakage_test,
        final_leakage_test_probabilities
    )
)


print(
    "Final Test calibration error: "
    f"{final_leakage_test_calibration_error:.2f} pp"
)

In [ ]:
# ================================================================
# 45.16 — FINAL TEST BOOTSTRAP
# ================================================================

def bootstrap_leakage_metrics(
    y_true,
    probabilities,
    n_bootstrap=2000,
    random_state=42
):

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )


    rng = np.random.default_rng(
        random_state
    )


    n = len(
        y_true
    )


    records = []


    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )


        y_sample = (
            y_true[
                indices
            ]
        )


        probability_sample = (
            probabilities[
                indices
            ]
        )


        if (
            np.unique(
                y_sample
            ).size
            < 2
        ):

            continue


        records.append(
            {
                "PR-AUC":
                    average_precision_score(
                        y_sample,
                        probability_sample
                    ),

                "ROC-AUC":
                    roc_auc_score(
                        y_sample,
                        probability_sample
                    ),

                "Brier Score":
                    brier_score_loss(
                        y_sample,
                        probability_sample
                    ),

                "Lift @ 10%":
                    lift_at_fraction(
                        y_sample,
                        probability_sample,
                        0.10
                    ),

                "Lift @ 20%":
                    lift_at_fraction(
                        y_sample,
                        probability_sample,
                        0.20
                    ),
            }
        )


    return pd.DataFrame(
        records
    )

In [ ]:
final_leakage_test_bootstrap = (
    bootstrap_leakage_metrics(
        y_true=y_leakage_test,
        probabilities=final_leakage_test_probabilities,
        n_bootstrap=2000,
        random_state=42
    )
)

In [ ]:
# ================================================================
# 45.17 — FINAL TEST UNCERTAINTY
# ================================================================

final_leakage_test_bootstrap_summary = []


for metric in [
    "PR-AUC",
    "ROC-AUC",
    "Brier Score",
    "Lift @ 10%",
    "Lift @ 20%",
]:

    final_leakage_test_bootstrap_summary.append(
        {
            "Metric":
                metric,

            "Estimate":
                final_leakage_test_results[
                    metric
                ].iloc[0],

            "Bootstrap Mean":
                final_leakage_test_bootstrap[
                    metric
                ].mean(),

            "CI 2.5%":
                final_leakage_test_bootstrap[
                    metric
                ].quantile(
                    0.025
                ),

            "CI 97.5%":
                final_leakage_test_bootstrap[
                    metric
                ].quantile(
                    0.975
                ),
        }
    )


final_leakage_test_bootstrap_summary = pd.DataFrame(
    final_leakage_test_bootstrap_summary
)


display(
    final_leakage_test_bootstrap_summary
)

In [ ]:
# ================================================================
# 45.18 — DEVELOPMENT → TEST FEATURE DRIFT
# ================================================================

final_leakage_drift_records = []


for feature in leakage_numeric_features:

    development_values = pd.to_numeric(
        X_final_leakage_development[
            feature
        ],
        errors="coerce"
    )


    test_values = pd.to_numeric(
        X_leakage_test[
            feature
        ],
        errors="coerce"
    )


    development_mean = (
        development_values.mean()
    )


    test_mean = (
        test_values.mean()
    )


    pooled_std = np.sqrt(
        (
            development_values.std() ** 2
            +
            test_values.std() ** 2
        )
        /
        2
    )


    if (
        pd.isna(
            pooled_std
        )
        or
        pooled_std == 0
    ):

        smd = np.nan

    else:

        smd = (
            test_mean
            -
            development_mean
        ) / pooled_std


    final_leakage_drift_records.append(
        {
            "feature":
                feature,

            "development_mean":
                development_mean,

            "test_mean":
                test_mean,

            "SMD":
                smd,

            "absolute_SMD":
                abs(
                    smd
                )
                if not pd.isna(
                    smd
                )
                else np.nan,
        }
    )


leakage_development_test_drift = (
    pd.DataFrame(
        final_leakage_drift_records
    )
    .sort_values(
        "absolute_SMD",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    leakage_development_test_drift.head(15)
)

### Step 45 Summary — Final Untouched Revenue Leakage Test

Step 45 performed the first and only evaluation of the frozen Full Gradient Boosting model on the future Test population.

The model architecture, features, hyperparameters and preprocessing rules had already been frozen before Test outcomes were inspected.

### Final Test Population

The final development population contained **23,617 historical prediction episodes** whose 90-day target windows had fully matured before the July 2011 Test period began.

The development prediction history extended from **May 2010 through March 2011**, with the latest admissible outcome completing on **29 June 2011**.

The Test population contained July–August 2011 prediction episodes.

Test leakage prevalence was approximately **43.27%**.

### Final Out-of-Time Performance

The frozen Gradient Boosting model achieved approximately:

- **PR-AUC: 0.625**
- **ROC-AUC: 0.713**
- **Brier score: 0.214**
- **Lift@10%: 1.66×**
- **Lift@20%: 1.54×**
- **Capture@10%: 16.61%**
- **Capture@20%: 30.81%**

The model therefore retained strong future discrimination and commercially meaningful concentration of leakage risk.

### PR-AUC Remained Strong Relative to Test Prevalence

Raw PR-AUC declined from approximately **0.654 during Validation to 0.625 during Test**.

However, leakage prevalence also declined from approximately **50.1% to 43.3%**.

PR-AUC relative to prevalence improved from approximately **1.31× to 1.44×**.

Normalised PR-AUC also improved from approximately **0.306 to 0.338**.

The lower raw PR-AUC therefore does not indicate deterioration in relative model quality.

### ROC Discrimination Improved

ROC-AUC increased from approximately **0.693 during Validation to 0.713 during Test**.

The bootstrap 95% confidence interval was approximately:

> **0.697 to 0.728**

The model therefore demonstrates robust out-of-time ranking ability.

### Commercial Lift Strengthened

Top-decile lift improved from approximately **1.42× to 1.66×**.

Top-20% lift improved from approximately **1.40× to 1.54×**.

Bootstrap confidence intervals remained completely above random targeting:

- **Lift@10%: approximately 1.55×–1.75×**
- **Lift@20%: approximately 1.47×–1.61×**

This provides strong evidence that the model can materially improve customer prioritisation compared with untargeted intervention.

### Strong Score-Segment Separation

The highest-risk Test decile experienced approximately **71.8% leakage**, compared with overall leakage of approximately **43.3%**.

The lowest-risk decile experienced only approximately **11.7% leakage**.

The model therefore generates approximately a **60 percentage-point difference** between the highest- and lowest-risk ends of the score distribution.

This is highly meaningful operational separation.

### Calibration Remained Strong

Final Test mean absolute calibration error was approximately **3.43 percentage points**, compared with approximately **4.03 percentage points during Validation**.

Predicted and observed leakage rates therefore remained closely aligned.

This validates the selection of Gradient Boosting over the less well-calibrated Logistic Regression challenger.

### Generalisation Survived Substantial Population Drift

Several time-sensitive variables changed materially between development and Test.

Absolute SMD exceeded 1.0 for:

- cyclical calendar variables;
- prior completed leakage count;
- and prior leakage-event count.

Despite this population shift, the frozen model retained strong discrimination, lift and calibration.

This provides meaningful evidence that the revenue-leakage signal generalises beyond the original development distribution.

## Final Predictive Conclusion

The frozen Full Gradient Boosting model successfully passed its untouched out-of-time Test.

The model demonstrates:

- robust temporal discrimination;
- strong concentration of high-risk customers;
- statistically credible commercial lift;
- good probability calibration;
- and resilience under substantial population drift.

The predictive model therefore supports progression to monitored operational use.

The next layer converts leakage probability into an economically prioritised **Expected Revenue Leakage Exposure** score.



## STEP 46 — EXPECTED REVENUE LEAKAGE EXPOSURE
This is the part that makes Notebook 4 materially different from an ordinary churn model.

A customer can have:
- high leakage probability but very little money at risk;
- moderate leakage probability but substantial revenue at risk.

So I should not rank customers by probability alone.

I will create an economic prioritisation layer.

In [ ]:
# ================================================================
# 46.1 — FINAL MODEL DECISION
# ================================================================

FINAL_LEAKAGE_DECISION = "GO"


print(
    f"FINAL REVENUE LEAKAGE MODEL DECISION: "
    f"{FINAL_LEAKAGE_DECISION}"
)

## Final Deployment Decision — GO

The Revenue Leakage model receives a final decision of:

> **GO**

GO means that the frozen model has demonstrated sufficiently strong out-of-time predictive performance to support monitored operational use.

The model should be used to prioritise customers for:

- retention attention;
- account review;
- proactive communication;
- customer-success intervention;
- and controlled retention experiments.

GO does not mean that every predicted leakage event will occur.

The model estimates future leakage risk; it does not prove that intervention will prevent leakage or that any specific treatment is profitable.

In [ ]:
# ================================================================
# 46.3 — FINAL TEST ECONOMIC SCORING TABLE
# ================================================================

leakage_test_scoring = (
    leakage_test_data[
        [
            "customer",
            "prediction_date",
            "prediction_period",
            "baseline_revenue",
        ]
    ]
    .copy()
)


leakage_test_scoring[
    "leakage_probability"
] = (
    final_leakage_test_probabilities
)


display(
    leakage_test_scoring.head()
)

In [ ]:
# ================================================================
# 46.4 — PROBABILITY RANK
# ================================================================

leakage_test_scoring[
    "probability_rank"
] = (
    leakage_test_scoring[
        "leakage_probability"
    ]
    .rank(
        method="first",
        ascending=False
    )
)


leakage_test_scoring[
    "probability_percentile"
] = (
    leakage_test_scoring[
        "leakage_probability"
    ]
    .rank(
        pct=True
    )
)

### 46.5 — Conservative minimum economic exposure
Because the target requires:
> future revenue ≤ 50% of baseline,

a positive event implies a decline of at least 50%.

So a conservative economically interpretable quantity is:
>probability × 50% × baseline revenue

In [ ]:
# ================================================================
# 46.5 — CONSERVATIVE EXPECTED LEAKAGE EXPOSURE
# ================================================================

MINIMUM_EVENT_LOSS_RATE = 0.50


leakage_test_scoring[
    "minimum_expected_leakage_exposure"
] = (
    leakage_test_scoring[
        "leakage_probability"
    ]
    *
    MINIMUM_EVENT_LOSS_RATE
    *
    leakage_test_scoring[
        "baseline_revenue"
    ]
)

This is deliberately conservative.

It does not claim that this is the exact expected financial loss.

It means:
> expected minimum revenue exposure implied by the model's event probability and target definition.

In [ ]:
# ================================================================
# 46.6 — EXPOSURE RANK
# ================================================================

leakage_test_scoring[
    "exposure_rank"
] = (
    leakage_test_scoring[
        "minimum_expected_leakage_exposure"
    ]
    .rank(
        method="first",
        ascending=False
    )
)


leakage_test_scoring[
    "exposure_percentile"
] = (
    leakage_test_scoring[
        "minimum_expected_leakage_exposure"
    ]
    .rank(
        pct=True
    )
)

In [ ]:
# ================================================================
# 46.7 — PROBABILITY VS ECONOMIC PRIORITY
# ================================================================

ranking_relationship = (
    leakage_test_scoring[
        [
            "leakage_probability",
            "baseline_revenue",
            "minimum_expected_leakage_exposure",
        ]
    ]
    .corr(
        method="spearman"
    )
)


display(
    ranking_relationship
)

In [ ]:
# ================================================================
# 46.8 — ECONOMIC PRIORITY BANDS
# ================================================================

leakage_test_scoring[
    "economic_priority_band"
] = pd.qcut(
    leakage_test_scoring[
        "minimum_expected_leakage_exposure"
    ],
    q=5,
    labels=[
        "P5 — Lowest Exposure",
        "P4",
        "P3",
        "P2",
        "P1 — Highest Exposure",
    ],
    duplicates="drop"
)

In [ ]:
# ================================================================
# 46.9 — RETROSPECTIVE ECONOMIC EVALUATION
# ================================================================

leakage_test_scoring[
    "actual_leakage_event"
] = (
    y_leakage_test
    .to_numpy()
)


test_economic_outcomes = (
    leakage_feature_table.loc[
        leakage_feature_table[
            "prediction_period"
        ]
        .between(
            TEST_START,
            TEST_END
        ),
        [
            "customer",
            "prediction_date",
            "revenue_leakage_amount",
            "future_revenue",
        ]
    ]
    .copy()
)


leakage_test_scoring = (
    leakage_test_scoring
    .merge(
        test_economic_outcomes,
        on=[
            "customer",
            "prediction_date"
        ],
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
# ================================================================
# 46.10 — ECONOMIC PRIORITY PERFORMANCE
# ================================================================

economic_priority_summary = (
    leakage_test_scoring
    .groupby(
        "economic_priority_band",
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        average_probability=(
            "leakage_probability",
            "mean"
        ),

        median_baseline_revenue=(
            "baseline_revenue",
            "median"
        ),

        average_expected_exposure=(
            "minimum_expected_leakage_exposure",
            "mean"
        ),

        observed_leakage_rate=(
            "actual_leakage_event",
            "mean"
        ),

        total_actual_leakage=(
            "revenue_leakage_amount",
            "sum"
        ),

        average_actual_leakage=(
            "revenue_leakage_amount",
            "mean"
        ),
    )
)


economic_priority_summary[
    "observed_leakage_rate_pct"
] = (
    economic_priority_summary[
        "observed_leakage_rate"
    ]
    *
    100
)


display(
    economic_priority_summary
)

In [ ]:
# ================================================================
# 46.11 — ACTUAL REVENUE LEAKAGE CAPTURE
# ================================================================

economic_ranked_test = (
    leakage_test_scoring
    .sort_values(
        "minimum_expected_leakage_exposure",
        ascending=False
    )
    .reset_index(drop=True)
)


total_actual_test_leakage = (
    economic_ranked_test[
        "revenue_leakage_amount"
    ]
    .sum()
)


economic_capture_records = []


for fraction in [
    0.10,
    0.20,
    0.30,
]:

    n_top = int(
        np.ceil(
            len(
                economic_ranked_test
            )
            *
            fraction
        )
    )


    top_actual_leakage = (
        economic_ranked_test
        .head(
            n_top
        )[
            "revenue_leakage_amount"
        ]
        .sum()
    )


    economic_capture_records.append(
        {
            "Customer Fraction":
                fraction,

            "Episodes":
                n_top,

            "Actual Leakage Captured":
                top_actual_leakage,

            "Share of Actual Leakage Captured":
                (
                    top_actual_leakage
                    /
                    total_actual_test_leakage
                ),

            "Share of Actual Leakage Captured (%)":
                (
                    top_actual_leakage
                    /
                    total_actual_test_leakage
                    *
                    100
                ),
        }
    )


economic_leakage_capture = pd.DataFrame(
    economic_capture_records
)


display(
    economic_leakage_capture
)

In [ ]:
# ================================================================
# 46.12 — PROBABILITY VS EXPOSURE TARGETING
# ================================================================

def financial_capture_at_fraction(
    scoring_data,
    score_column,
    financial_outcome_column,
    fraction
):

    ranked = (
        scoring_data
        .sort_values(
            score_column,
            ascending=False
        )
        .reset_index(drop=True)
    )


    n_top = int(
        np.ceil(
            len(
                ranked
            )
            *
            fraction
        )
    )


    total_financial_outcome = (
        ranked[
            financial_outcome_column
        ]
        .sum()
    )


    captured = (
        ranked
        .head(
            n_top
        )[
            financial_outcome_column
        ]
        .sum()
    )


    return (
        captured
        /
        total_financial_outcome
    )

In [ ]:
financial_targeting_comparison = []


for fraction in [
    0.10,
    0.20,
    0.30,
]:

    probability_capture = (
        financial_capture_at_fraction(
            scoring_data=leakage_test_scoring,
            score_column="leakage_probability",
            financial_outcome_column="revenue_leakage_amount",
            fraction=fraction
        )
    )


    exposure_capture = (
        financial_capture_at_fraction(
            scoring_data=leakage_test_scoring,
            score_column="minimum_expected_leakage_exposure",
            financial_outcome_column="revenue_leakage_amount",
            fraction=fraction
        )
    )


    financial_targeting_comparison.append(
        {
            "Target Fraction":
                fraction,

            "Probability Ranking Capture":
                probability_capture,

            "Exposure Ranking Capture":
                exposure_capture,

            "Exposure Ranking Advantage":
                (
                    exposure_capture
                    -
                    probability_capture
                ),
        }
    )


financial_targeting_comparison = pd.DataFrame(
    financial_targeting_comparison
)


display(
    financial_targeting_comparison
)

In [ ]:
# ================================================================
# 46.13 — RISK × VALUE MATRIX
# ================================================================

leakage_test_scoring[
    "risk_band"
] = pd.qcut(
    leakage_test_scoring[
        "leakage_probability"
    ],
    q=3,
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk",
    ],
    duplicates="drop"
)


leakage_test_scoring[
    "value_band"
] = pd.qcut(
    leakage_test_scoring[
        "baseline_revenue"
    ],
    q=3,
    labels=[
        "Low Value",
        "Medium Value",
        "High Value",
    ],
    duplicates="drop"
)


risk_value_matrix = (
    leakage_test_scoring
    .groupby(
        [
            "risk_band",
            "value_band",
        ],
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        observed_leakage_rate=(
            "actual_leakage_event",
            "mean"
        ),

        average_baseline_revenue=(
            "baseline_revenue",
            "mean"
        ),

        total_actual_leakage=(
            "revenue_leakage_amount",
            "sum"
        ),

        average_expected_exposure=(
            "minimum_expected_leakage_exposure",
            "mean"
        ),
    )
)


risk_value_matrix[
    "observed_leakage_rate_pct"
] = (
    risk_value_matrix[
        "observed_leakage_rate"
    ]
    *
    100
)


display(
    risk_value_matrix
)

## Revenue Leakage Operating Architecture

The final deployment architecture should separate two questions.

### 1. Who Is Most Likely to Leak?

Answered by:

> **Leakage Probability**

This is useful for understanding behavioural risk.

### 2. Where Is the Most Revenue at Risk?

Answered by:

> **Expected Leakage Exposure**

The initial conservative exposure score is:

> **Leakage Probability × 50% × Prior 90-Day Revenue**

The 50% factor reflects the minimum loss implied by the model's leakage-event definition.

The result is not an accounting forecast.

It is a customer-prioritisation score expressed in economically meaningful units.

### Strategic Customer Matrix

Customers can then be separated into four broad strategic groups:

**High Risk / High Value**

Highest intervention priority.

These customers combine elevated probability of deterioration with material revenue exposure.

**Low Risk / High Value**

Protective relationship management.

These customers are valuable but currently show relatively low predicted deterioration risk.

**High Risk / Low Value**

Automated or low-cost retention treatment.

Their probability of leakage may be high, but expensive intervention may not be economically justified.

**Low Risk / Low Value**

Lowest immediate retention priority.

Intervention should generally remain low-cost unless other strategic considerations apply.

This architecture prevents the business from confusing customer risk with customer economic importance.

In [ ]:
# ================================================================
# 46.15 — FINAL REVENUE LEAKAGE MODEL CARD
# ================================================================

final_leakage_model_card = pd.DataFrame(
    {
        "Field": [
            "Business problem",
            "Prediction unit",
            "Prediction point",
            "Baseline window",
            "Target",
            "Outcome horizon",
            "Final model",
            "Final Test episodes",
            "Final Test prevalence",
            "Final PR-AUC",
            "Final ROC-AUC",
            "Final Brier score",
            "Final Lift@10%",
            "Final Lift@20%",
            "Final Capture@10%",
            "Final Capture@20%",
            "Final calibration error",
            "Final decision",
            "Primary risk score",
            "Economic priority score",
            "Primary limitation",
        ],

        "Final Position": [
            "Identify active customers at risk of substantial future revenue deterioration",

            "One active customer at one month-end prediction point",

            "Month-end",

            "Previous 90 days",

            "Future 90-day revenue <= 50% of prior 90-day revenue",

            "90 days",

            "Full Gradient Boosting",

            len(
                leakage_test_data
            ),

            test_leakage_prevalence,

            final_leakage_test_results[
                "PR-AUC"
            ].iloc[0],

            final_leakage_test_results[
                "ROC-AUC"
            ].iloc[0],

            final_leakage_test_results[
                "Brier Score"
            ].iloc[0],

            final_leakage_test_results[
                "Lift @ 10%"
            ].iloc[0],

            final_leakage_test_results[
                "Lift @ 20%"
            ].iloc[0],

            final_leakage_test_results[
                "Capture @ 10%"
            ].iloc[0],

            final_leakage_test_results[
                "Capture @ 20%"
            ].iloc[0],

            final_leakage_test_calibration_error,

            FINAL_LEAKAGE_DECISION,

            "Predicted probability of material 90-day revenue leakage",

            "Probability × 50% × prior 90-day revenue",

            (
                "Prediction identifies revenue deterioration risk, "
                "not causal preventability or guaranteed recoverable revenue"
            ),
        ],
    }
)


display(
    final_leakage_model_card
)

In [ ]:
# ================================================================
# 46.16 — FINAL LIMITATIONS REGISTER
# ================================================================

final_leakage_limitations = pd.DataFrame(
    [
        {
            "Limitation":
                "Leakage probability is not causal preventability",

            "Implication":
                "The model cannot determine whether intervention will prevent the predicted deterioration."
        },

        {
            "Limitation":
                "Exposure score is a prioritisation proxy",

            "Implication":
                "Probability × 50% × baseline revenue is not an accounting forecast of actual loss."
        },

        {
            "Limitation":
                "Overlapping prediction windows",

            "Implication":
                "The same customer may appear in multiple monthly prediction episodes."
        },

        {
            "Limitation":
                "Strong calendar dependence",

            "Implication":
                "Seasonal relationships should be monitored as future trading patterns evolve."
        },

        {
            "Limitation":
                "Accumulated history changes through time",

            "Implication":
                "Prior leakage-history variables require ongoing drift monitoring."
        },

        {
            "Limitation":
                "Identified customers only",

            "Implication":
                "Customer-level modelling excludes revenue that cannot be linked to a customer identifier."
        },

        {
            "Limitation":
                "Gross revenue focus",

            "Implication":
                "Revenue exposure does not necessarily equal profit or contribution margin exposure."
        },

        {
            "Limitation":
                "No treatment-cost optimisation",

            "Implication":
                "Campaign economics must be established through controlled experiments."
        },
    ]
)


display(
    final_leakage_limitations
)

In [ ]:
# ================================================================
# 46.17 — PRODUCTION MONITORING
# ================================================================

leakage_monitoring_framework = pd.DataFrame(
    [
        {
            "Area":
                "Target prevalence",
            "Metric":
                "90-day material leakage rate"
        },

        {
            "Area":
                "Discrimination",
            "Metric":
                "PR-AUC and ROC-AUC"
        },

        {
            "Area":
                "Commercial ranking",
            "Metric":
                "Lift@10%, Lift@20%, event capture"
        },

        {
            "Area":
                "Financial prioritisation",
            "Metric":
                "Actual leakage captured by top exposure-ranked customers"
        },

        {
            "Area":
                "Calibration",
            "Metric":
                "Predicted vs observed leakage probability"
        },

        {
            "Area":
                "Feature drift",
            "Metric":
                "SMD / PSI"
        },

        {
            "Area":
                "Score drift",
            "Metric":
                "Probability and exposure-score distributions"
        },

        {
            "Area":
                "Intervention effect",
            "Metric":
                "Treatment-control incremental revenue retention"
        },

        {
            "Area":
                "Economics",
            "Metric":
                "Incremental margin recovered relative to intervention cost"
        },
    ]
)


display(
    leakage_monitoring_framework
)

In [ ]:
# ================================================================
# 46.18 — NOTEBOOK 4 GOVERNANCE LOCK
# ================================================================

LEAKAGE_TEST_SET_CONSUMED = True
NOTEBOOK_4_COMPLETE = True


assert LEAKAGE_MODEL_FROZEN is True
assert LEAKAGE_TEST_SET_OPENED is True
assert LEAKAGE_TEST_SET_CONSUMED is True
assert FINAL_LEAKAGE_DECISION == "GO"


print(
    "✓ Revenue Leakage model frozen."
)

print(
    "✓ Untouched temporal Test completed."
)

print(
    "✓ Final decision: GO."
)

print(
    "✓ Economic exposure architecture created."
)

print(
    "✓ Test population permanently retired from model development."
)

print(
    "✓ Notebook 4 complete."
)

### Step 46 Summary — Expected Revenue Leakage Exposure

Step 46 translated predicted customer leakage risk into an economically prioritised revenue-risk architecture.

The predictive model answers:

> **Who is likely to experience substantial revenue deterioration?**

The economic exposure layer answers:

> **Where is the most revenue actually at risk?**

### Leakage Probability and Customer Value Are Different Dimensions

Predicted leakage probability and baseline customer revenue showed a moderate negative Spearman relationship of approximately **−0.45**.

Higher-value customers therefore tended, on average, to have lower predicted leakage probability.

However, baseline revenue was strongly related to Expected Revenue Leakage Exposure, with correlation of approximately **0.91**.

Leakage probability and the final exposure score themselves had very little rank correlation, at approximately **−0.09**.

This demonstrates that probability ranking and financial-exposure ranking represent materially different business objectives.

### Economic Exposure Ranking Strongly Concentrates Actual Leakage

Customers were ranked using the conservative score:

> **Leakage Probability × 50% × Prior 90-Day Revenue**

The 50% factor represents the minimum deterioration implied by the primary leakage-event definition.

When ranked by this exposure measure:

- the highest **10%** of customer episodes captured approximately **47.98%** of actual observed revenue leakage;
- the highest **20%** captured approximately **61.84%**;
- the highest **30%** captured approximately **71.95%**.

The revenue-leakage problem is therefore highly concentrated economically.

### Probability-Only Targeting Is Financially Inferior

Ranking customers purely by predicted leakage probability captured only approximately:

- **14.47%** of actual leakage in the top 10%;
- **23.96%** in the top 20%;
- **36.20%** in the top 30%.

By comparison, economic exposure ranking captured approximately:

- **47.98%**;
- **61.84%**;
- **71.95%**.

The exposure-ranking advantage was therefore approximately:

- **+33.5 percentage points** at 10% customer coverage;
- **+37.9 percentage points** at 20%;
- **+35.8 percentage points** at 30%.

This is one of the most important findings of the notebook.

A customer can have high behavioural leakage risk while representing relatively little financial exposure.

Conversely, a customer with only moderate leakage probability may represent substantial revenue at risk because of their economic scale.

### Final Operating Principle

Revenue Leakage management should therefore use two distinct scores:

**Leakage Probability**

Used to understand behavioural deterioration risk.

**Expected Revenue Leakage Exposure**

Used to allocate scarce retention resources according to economic consequence.

The final operating principle is:

> **Do not prioritise only who is most likely to leak. Prioritise where the most revenue is at risk.**

The initial exposure score is a prioritisation mechanism rather than an accounting loss forecast.

Its purpose is to direct retention resources toward customers where predicted deterioration and economic consequence intersect.

In [ ]:
# ================================================================
# 46.19 — FINAL NOTEBOOK 4 COMPLETION
# ================================================================

assert LEAKAGE_MODEL_FROZEN is True
assert LEAKAGE_TEST_SET_OPENED is True
assert LEAKAGE_TEST_SET_CONSUMED is True
assert FINAL_LEAKAGE_DECISION == "GO"

assert (
    financial_targeting_comparison[
        "Exposure Ranking Capture"
    ]
    >
    financial_targeting_comparison[
        "Probability Ranking Capture"
    ]
).all()


NOTEBOOK_4_COMPLETE = True


print(
    "✓ Predictive Revenue Leakage model validated."
)

print(
    "✓ Final decision: GO."
)

print(
    "✓ Expected Revenue Leakage Exposure architecture validated."
)

print(
    "✓ Economic ranking materially outperforms probability-only ranking."
)

print(
    "✓ Notebook 4 complete."
)

# Final Notebook Conclusion

## Customer Revenue Leakage Risk

This notebook investigated whether active customers can be identified before experiencing substantial future revenue deterioration.

A leakage event was defined as:

> **future 90-day revenue falling to 50% or less of the preceding 90-day revenue baseline**

This captures both:

- complete future inactivity; and
- material contraction among customers who remain active.

## Predictive Signal

Revenue leakage was meaningfully predictable from:

- prior-period revenue;
- current-versus-prior revenue momentum;
- purchase frequency;
- customer activity;
- recency;
- product breadth;
- historical purchasing behaviour;
- and seasonal context.

## Model Selection

Several modelling approaches were compared, including:

- prevalence and recency baselines;
- Logistic Regression;
- Random Forest;
- Gradient Boosting;
- HistGradientBoosting;
- and stable-core feature challengers.

The strongest models were close on PR-AUC.

Full Gradient Boosting was ultimately selected because it provided the strongest overall balance of:

- predictive discrimination;
- high-risk customer concentration;
- probability calibration;
- temporal robustness;
- and suitability for economic exposure scoring.

## Final Out-of-Time Test

On the untouched future Test population, the frozen Gradient Boosting model achieved approximately:

- **PR-AUC: 0.625**
- **ROC-AUC: 0.713**
- **Brier score: 0.214**
- **Lift@10%: 1.66×**
- **Lift@20%: 1.54×**
- **Capture@10%: 16.61%**
- **Capture@20%: 30.81%**
- **Calibration error: 3.43 percentage points**

Bootstrap confidence intervals for both Lift@10% and Lift@20% remained entirely above random targeting.

The model therefore demonstrated credible out-of-time generalisation.

## Strong Customer Separation

Overall Test leakage prevalence was approximately **43.3%**.

The highest-risk decile experienced approximately:

> **71.8% leakage**

while the lowest-risk decile experienced only approximately:

> **11.7% leakage**

This represents roughly a **60-percentage-point separation** between the highest- and lowest-risk customer groups.

## Revenue Risk Is Not the Same as Revenue Exposure

One of the most important findings of the notebook is that customers with the highest probability of leakage are not necessarily the customers with the greatest financial exposure.

Predicted leakage probability and baseline revenue had a Spearman relationship of approximately:

> **−0.45**

This means higher-value customers tended, on average, to have lower leakage probability.

Probability and Expected Revenue Leakage Exposure had almost no rank relationship:

> **approximately −0.09**

Risk ranking and economic ranking therefore represent materially different business objectives.

## Expected Revenue Leakage Exposure

A conservative economic priority score was created as:

> **Leakage Probability × 50% × Prior 90-Day Revenue**

The 50% factor represents the minimum deterioration implied by the primary leakage definition.

This score is not an accounting forecast.

It is an economically meaningful customer-prioritisation measure.

## Economic Ranking Materially Outperformed Probability Ranking

When customers were ranked by Expected Revenue Leakage Exposure:

- the highest **10%** captured approximately **47.98%** of actual observed leakage;
- the highest **20%** captured approximately **61.84%**;
- the highest **30%** captured approximately **71.95%**.

By comparison, probability-only ranking captured approximately:

- **14.47%** at the top 10%;
- **23.96%** at the top 20%;
- **36.20%** at the top 30%.

Economic exposure ranking therefore improved actual leakage capture by approximately:

- **+33.5 percentage points** at 10% coverage;
- **+37.9 percentage points** at 20%;
- **+35.8 percentage points** at 30%.

This demonstrates that customer retention resources should not be allocated on leakage probability alone.

## Final Operating Principle

The final architecture uses two distinct decision measures:

### Leakage Probability

Answers:

> **Who is most likely to experience substantial future revenue deterioration?**

### Expected Revenue Leakage Exposure

Answers:

> **Where is the most revenue economically at risk?**

The key operating principle is:

> **Do not prioritise only who is most likely to leak. Prioritise where the most revenue is at risk.**

## Final Decision

> **GO**

The Revenue Leakage model is suitable for monitored operational use as a customer-risk and revenue-exposure prioritisation layer.

It predicts future revenue deterioration.

It does not prove that intervention will prevent the deterioration, that all predicted revenue exposure is recoverable, or that a specific retention treatment will generate positive incremental ROI.

Controlled treatment experiments are therefore required before converting predicted risk into claims of retained revenue or causal campaign impact.

---

**Notebook 4 Status: COMPLETE**

In [ ]:
# ================================================================
# EXPORT AUDIT — REVENUE LEAKAGE NOTEBOOK
# ================================================================

leakage_export_candidates = []


search_terms = [
    "leak",
    "exposure",
    "prob",
    "score",
    "prediction",
    "pred",
]


for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    columns = list(
        obj.columns
    )


    columns_lower = [
        str(column).lower()
        for column in columns
    ]


    has_customer = (
        "customer"
        in columns_lower
    )


    matching_columns = [
        column
        for column in columns
        if any(
            term
            in str(column).lower()
            for term in search_terms
        )
    ]


    if (
        has_customer
        and
        matching_columns
    ):

        leakage_export_candidates.append(
            {
                "DataFrame":
                    name,

                "Rows":
                    len(obj),

                "Matching Columns":
                    matching_columns,

                "All Columns":
                    columns,
            }
        )


leakage_export_candidates = pd.DataFrame(
    leakage_export_candidates
)


display(
    leakage_export_candidates
)

In [ ]:
# ================================================================
# EXPORT AUDIT 2 — LEAKAGE TEST SCORING STRUCTURE
# ================================================================

print(
    "leakage_test_scoring shape:",
    leakage_test_scoring.shape
)

print(
    "\nColumns:"
)

for column in leakage_test_scoring.columns:

    print(
        " -",
        column
    )


display(
    leakage_test_scoring.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT AUDIT 3 — ECONOMIC RANKED TEST STRUCTURE
# ================================================================

print(
    "economic_ranked_test shape:",
    economic_ranked_test.shape
)

print(
    "\nColumns:"
)

for column in economic_ranked_test.columns:

    print(
        " -",
        column
    )


display(
    economic_ranked_test.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT — REVENUE LEAKAGE RISK MODEL OUTPUT
# ================================================================

from pathlib import Path

MODEL_OUTPUT_DIR = Path("model_outputs")

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------
# Construct clean historical scoring output
# ------------------------------------------------

revenue_leakage_scores = (
    leakage_test_scoring[
        [
            "customer",
            "prediction_date",
            "prediction_period",
            "baseline_revenue",
            "leakage_probability",
            "probability_percentile",
            "minimum_expected_leakage_exposure",
            "exposure_percentile",
            "economic_priority_band",
            "risk_band",
            "value_band",
        ]
    ]
    .copy()
)


# ------------------------------------------------
# Canonical Notebook 8 naming
# ------------------------------------------------

revenue_leakage_scores = (
    revenue_leakage_scores
    .rename(
        columns={
            "minimum_expected_leakage_exposure":
                "leakage_exposure_score",

            "probability_percentile":
                "leakage_probability_percentile",

            "exposure_percentile":
                "leakage_exposure_percentile",
        }
    )
)


# ------------------------------------------------
# Standardise customer identifier
# ------------------------------------------------

revenue_leakage_scores[
    "customer"
] = (
    pd.to_numeric(
        revenue_leakage_scores[
            "customer"
        ],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


revenue_leakage_scores[
    "prediction_date"
] = pd.to_datetime(
    revenue_leakage_scores[
        "prediction_date"
    ]
)


# ------------------------------------------------
# Explicit model provenance
# ------------------------------------------------

revenue_leakage_scores[
    "leakage_model_scope"
] = (
    "Risk of future 90-day merchandise revenue "
    "falling to 50% or less of prior 90-day baseline"
)


revenue_leakage_scores[
    "leakage_exposure_definition"
] = (
    "Conservative prioritisation proxy: "
    "leakage probability × 50% × baseline revenue; "
    "not calibrated expected accounting loss"
)


revenue_leakage_scores[
    "leakage_score_provenance"
] = (
    "historical held-out Test customer-month scoring"
)


# ------------------------------------------------
# Safety checks
# ------------------------------------------------

assert (
    revenue_leakage_scores[
        "customer"
    ]
    .notna()
    .all()
)


assert (
    revenue_leakage_scores[
        "prediction_date"
    ]
    .notna()
    .all()
)


assert (
    revenue_leakage_scores[
        "leakage_probability"
    ]
    .between(
        0,
        1
    )
    .all()
)


assert (
    revenue_leakage_scores[
        "leakage_probability_percentile"
    ]
    .between(
        0,
        1
    )
    .all()
)


assert (
    revenue_leakage_scores[
        "leakage_exposure_percentile"
    ]
    .between(
        0,
        1
    )
    .all()
)


assert (
    revenue_leakage_scores[
        [
            "customer",
            "prediction_date",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
), (
    "Expected one leakage score per customer "
    "per prediction date."
)


# ------------------------------------------------
# Save persistent output
# ------------------------------------------------

leakage_output_path = (
    MODEL_OUTPUT_DIR
    /
    "revenue_leakage_scores.parquet"
)


revenue_leakage_scores.to_parquet(
    leakage_output_path,
    index=False
)


# ------------------------------------------------
# Export audit
# ------------------------------------------------

print(
    "✓ Revenue Leakage scores exported."
)

print(
    leakage_output_path.resolve()
)

print(
    f"Rows: "
    f"{len(revenue_leakage_scores):,}"
)

print(
    f"Customers: "
    f"{revenue_leakage_scores['customer'].nunique():,}"
)

print(
    f"Prediction date range: "
    f"{revenue_leakage_scores['prediction_date'].min()} "
    f"to "
    f"{revenue_leakage_scores['prediction_date'].max()}"
)

print(
    f"Leakage probability range: "
    f"{revenue_leakage_scores['leakage_probability'].min():.4f} "
    f"to "
    f"{revenue_leakage_scores['leakage_probability'].max():.4f}"
)

print(
    f"Exposure score range: "
    f"{revenue_leakage_scores['leakage_exposure_score'].min():,.2f} "
    f"to "
    f"{revenue_leakage_scores['leakage_exposure_score'].max():,.2f}"
)


display(
    revenue_leakage_scores.head(
        10
    )
)